# Modelado — arquitectura multi-horizonte (h=2, 3, 4)

Orquesta el encargo de `docs/modeling/plan_arquitectura_horizontes.md`. La lógica está en `src/modeling/`; este notebook carga, llama y muestra. Nada se escribe en `data/`.

## Fase 1: diagnóstico de errores

Se analizan las predicciones fuera de muestra ya guardadas del experimento progresivo (`validacion_temporal_compacta`, semilla 17, GPU): regresión `base_6` de `log1p(casos)` y clasificación `base_6_poblacion_2017`. El umbral calibrado de cada temporada es el que ese experimento eligió con temporadas anteriores. Las temporadas 2022–2024 forman el diagnóstico; 2021 solo sirvió como calibración inicial y 2025 tiene 3 positivos. Las definiciones (fase de la curva, terciles, episodios, antelación) están en el docstring del módulo y son **retrospectivas**: sirven para diagnosticar, no son reglas operativas.

**Caché.** Los entrenamientos de las fases 3 y 4 se guardan en `models/experimentos/arquitectura_horizontes/cache/` con una huella de los datos, `config.yaml` y el código de `src/modeling/`; si algo cambia, se recalculan solos. Para la corrida de cierre de una etapa se fuerza el reentreno completo con `DENGUEKAY_FORZAR_REENTRENO=1`.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from src.modeling import cache_experimentos as cache

# Caché de los entrenamientos (fases 3 y 4). Se invalida sola si cambian gold, silver,
# config.yaml o el código de src/modeling. La corrida de CIERRE de cada etapa va forzada:
#   $env:DENGUEKAY_FORZAR_REENTRENO = "1"; python -m jupyter nbconvert --execute ...
FORZAR_REENTRENO = cache.forzar_desde_entorno()
HUELLA = cache.huella_encargo()
print("forzar reentreno:", FORZAR_REENTRENO, "| huella:", HUELLA[:12])
from src.eda.estilo import aplicar_estilo
from src.modeling import diagnostico_arquitectura as d
from src.utils.paths import (FIGURAS_ARQUITECTURA, GOLD, METRICAS_ARQUITECTURA,
                             METRICAS_COMPACTAS, PREDICCIONES_COMPACTAS, SILVER_INTEGRADO)

aplicar_estilo()
pd.set_option("display.precision", 3)
reporte = json.loads(METRICAS_COMPACTAS.read_text(encoding="utf-8"))
predicciones = pd.read_csv(PREDICCIONES_COMPACTAS, dtype={"ubigeo": "string"})
# Etiquetas completas (solo lectura) para rachas, adyacencia y tamaño de distrito.
etiquetas = pd.read_csv(GOLD / f"{SILVER_INTEGRADO.stem}_h2_gold.csv", dtype={"ubigeo": "string"},
                        usecols=["ubigeo", "anio", "semana", "semana_inicio", "casos_Dengue", "brote"],
                        parse_dates=["semana_inicio"])

datos = d.agregar_alertas(d.predicciones_seleccionadas(predicciones, reporte))
datos["nivel"] = d.nivel_casos(datos.casos_Dengue)
datos["fase"] = d.fase_curva(datos)
datos = datos.merge(d.terciles_distrito(etiquetas), left_on="ubigeo", right_index=True, how="left")
diag = datos[datos.bloque.isin(d.BLOQUES_DIAGNOSTICO)].copy()
ALERTAS = ["alerta_cls_calibrada", "alerta_cls_050", "alerta_regresion", "alerta_persistencia"]
datos.groupby(["horizonte", "bloque"]).size().unstack(0)

forzar reentreno: True | huella: 0e2c4bac8c6e


horizonte,2,4
bloque,,
calendario_2025,3380,3380
temporada_2021,3445,3445
temporada_2022,3380,3380
temporada_2023,3380,3380
temporada_2024,3380,3380


### 1.1 ¿Dónde pierde el regresor frente a la persistencia?

Razón < 1: el XGBoost tiene menos error que la persistencia. `sesgo` = media de (predicho − real); `suma_predicha_sobre_real` < 1 indica subestimación total.

In [2]:
errores_temporada = d.desglose_errores(datos, ["horizonte", "bloque"])
errores_temporada

,horizonte,bloque,n,casos_medios,mae_modelo,mae_persistencia,razon_mae,rmse_modelo,rmse_persistencia,razon_rmse,sesgo_modelo,sesgo_persistencia,suma_predicha_sobre_real
0,2,calendario_2025,3380.0,0.330,0.301,0.285,1.055,1.022,0.981,1.042,0.086,0.026,1.262
1,2,temporada_2021,3445.0,0.860,0.515,0.507,1.016,3.243,3.143,1.032,-0.299,-0.050,0.652
2,2,temporada_2022,3380.0,3.761,1.944,1.959,0.992,8.037,6.036,1.332,0.091,0.024,1.024
3,2,temporada_2023,3380.0,22.121,10.687,10.837,0.986,56.377,49.474,1.140,-8.282,-0.455,0.626
4,2,temporada_2024,3380.0,10.698,5.639,4.956,1.138,23.848,15.757,1.514,2.152,0.450,1.201
5,4,calendario_2025,3380.0,0.330,0.433,0.324,1.334,1.661,1.168,1.422,0.249,0.077,1.754
6,4,temporada_2021,3445.0,0.860,0.679,0.674,1.007,4.728,4.612,1.025,-0.415,-0.101,0.518
7,4,temporada_2022,3380.0,3.761,3.074,2.846,1.080,16.463,8.536,1.929,0.734,0.052,1.195
8,4,temporada_2023,3380.0,22.121,14.249,17.680,0.806,69.383,80.185,0.865,-12.104,-1.053,0.453
9,4,temporada_2024,3380.0,10.698,9.401,7.848,1.198,40.443,25.455,1.589,4.128,1.046,1.386


In [3]:
errores_tercil = d.desglose_errores(diag, ["horizonte", "tercil_distrito"])
errores_tercil

,horizonte,tercil_distrito,n,casos_medios,mae_modelo,mae_persistencia,razon_mae,rmse_modelo,rmse_persistencia,razon_rmse,sesgo_modelo,sesgo_persistencia,suma_predicha_sobre_real
0,2,bajo,3432.0,0.645,0.476,0.468,1.016,1.976,1.967,1.005,-0.078,-2.914e-04,0.879
1,2,medio,3276.0,5.230,2.877,2.910,0.989,10.230,9.192,1.113,-0.525,8.547e-03,0.900
2,2,alto,3432.0,30.389,14.771,14.237,1.037,60.416,51.054,1.183,-5.368,1.107e-02,0.823
3,4,bajo,3432.0,0.645,0.627,0.633,0.990,2.782,2.696,1.032,-0.054,-2.914e-04,0.916
4,4,medio,3276.0,5.230,4.213,4.356,0.967,17.576,13.974,1.258,-0.388,1.557e-02,0.926
5,4,alto,3432.0,30.389,21.671,23.153,0.936,79.475,82.755,0.960,-6.707,3.089e-02,0.779


In [4]:
errores_fase = d.desglose_errores(diag, ["horizonte", "fase"])
d.guardar_figura(d.figura_errores(errores_fase, "fase", "Error de casos por fase de la curva regional, temporadas 2022–2024"),
                 FIGURAS_ARQUITECTURA, "errores_por_fase")
errores_fase

,horizonte,fase,n,casos_medios,mae_modelo,mae_persistencia,razon_mae,rmse_modelo,rmse_persistencia,razon_rmse,sesgo_modelo,sesgo_persistencia,suma_predicha_sobre_real
0,2,valle,4810.0,1.963,1.032,1.181,0.874,3.519,3.808,0.924,-0.363,0.024,0.815
1,2,subida,2210.0,11.276,5.166,5.402,0.956,17.369,18.321,0.948,-1.814,-2.948,0.839
2,2,pico,1560.0,41.313,21.383,17.101,1.250,83.862,64.222,1.306,-8.979,-4.347,0.783
3,2,bajada,1560.0,15.917,7.703,10.069,0.765,27.573,35.713,0.772,-0.417,8.491,0.974
4,4,valle,4810.0,1.963,1.254,1.715,0.731,4.637,6.361,0.729,-0.666,0.246,0.661
5,4,subida,2210.0,11.276,6.443,7.617,0.846,22.233,25.610,0.868,-3.490,-4.493,0.690
6,4,pico,1560.0,41.313,30.638,24.368,1.257,105.286,93.098,1.131,-10.360,-13.892,0.749
7,4,bajada,1560.0,15.917,14.268,21.029,0.678,52.057,75.969,0.685,1.669,19.598,1.105


In [5]:
errores_nivel = d.desglose_errores(diag, ["horizonte", "nivel"])
d.guardar_figura(d.figura_errores(errores_nivel, "nivel", "Error de casos por nivel de casos reales, temporadas 2022–2024"),
                 FIGURAS_ARQUITECTURA, "errores_por_nivel")
errores_nivel

,horizonte,nivel,n,casos_medios,mae_modelo,mae_persistencia,razon_mae,rmse_modelo,rmse_persistencia,razon_rmse,sesgo_modelo,sesgo_persistencia,suma_predicha_sobre_real
0,2,0,6001.0,0.000,0.183,0.215,0.851,0.618,0.883,0.701,0.183,0.215,NaN
1,2,1–4,1810.0,1.961,1.671,2.150,0.777,2.766,3.797,0.728,0.134,0.698,1.068
2,2,5–19,1229.0,10.024,5.762,6.403,0.900,7.892,9.438,0.836,-1.001,0.800,0.900
3,2,20–99,827.0,42.762,19.238,20.270,0.949,29.116,29.760,0.978,-3.133,1.397,0.927
4,2,≥100,273.0,265.231,126.886,110.586,1.147,210.440,175.023,1.202,-65.672,-16.945,0.752
5,4,0,6001.0,0.000,0.229,0.319,0.717,0.856,1.580,0.542,0.229,0.319,NaN
6,4,1–4,1810.0,1.961,2.118,3.140,0.675,4.601,7.380,0.623,0.510,1.673,1.260
7,4,5–19,1229.0,10.024,7.771,9.850,0.789,13.032,16.414,0.794,-0.493,3.022,0.951
8,4,20–99,827.0,42.762,30.599,34.799,0.879,57.743,58.443,0.988,-0.832,5.302,0.981
9,4,≥100,273.0,265.231,184.112,173.703,1.060,268.678,276.682,0.971,-93.328,-47.198,0.648


### 1.2 ¿Qué son los falsos positivos?

Adyacente: hay una semana `brote` real del mismo distrito a ±2 semanas (aviso temprano o tardío). Aislado: ninguna.

In [6]:
fp = pd.concat([d.falsos_positivos(diag, etiquetas, a) for a in ALERTAS], ignore_index=True)
fp_total = (fp.groupby(["alerta", "horizonte"])[["fp", "adyacentes", "aislados"]].sum()
              .assign(fraccion_adyacentes=lambda t: t.adyacentes / t.fp).reset_index())
d.guardar_figura(d.figura_falsos_positivos(fp), FIGURAS_ARQUITECTURA, "falsos_positivos")
fp_total

,alerta,horizonte,fp,adyacentes,aislados,fraccion_adyacentes
0,alerta_cls_050,2,611,287,324,0.470
1,alerta_cls_050,4,609,245,364,0.402
2,alerta_cls_calibrada,2,1407,614,793,0.436
3,alerta_cls_calibrada,4,1431,549,882,0.384
4,alerta_persistencia,2,417,373,44,0.894
5,alerta_persistencia,4,530,344,186,0.649
6,alerta_regresion,2,219,206,13,0.941
7,alerta_regresion,4,255,207,48,0.812


### 1.3 Calibración

Fiabilidad agregada de 2022–2024 (bins de 0,1) y métricas por temporada con la regla literal p ≥ 0,50 frente al umbral calibrado cronológicamente.

In [7]:
probabilidad = d.metricas_probabilidad(datos[datos.bloque.ne("temporada_2021")])
fiabilidad = {h: d.curva_fiabilidad(g.brote, g.probabilidad_brote) for h, g in diag.groupby("horizonte")}
d.guardar_figura(d.figura_fiabilidad(diag), FIGURAS_ARQUITECTURA, "fiabilidad")
probabilidad

,horizonte,bloque,positivos,prob_media,prevalencia,brier,vp_050,fp_050,fn_050,precision_050,recall_050,f1_050,auprc,vp_calibrada,fp_calibrada,fn_calibrada,precision_calibrada,recall_calibrada,f1_calibrada,umbral_calibrado
0,2,calendario_2025,3,0.036,8.876e-04,0.010,0,41,3,0.000,0.000,0.000,0.010,1,112,2,0.009,0.333,0.017,0.275
1,2,temporada_2022,431,0.118,1.275e-01,0.080,191,148,240,0.563,0.443,0.496,0.558,298,344,133,0.464,0.691,0.555,0.225
2,2,temporada_2023,1039,0.203,3.074e-01,0.102,621,76,418,0.891,0.598,0.715,0.881,897,216,142,0.806,0.863,0.834,0.200
3,2,temporada_2024,857,0.301,2.536e-01,0.119,633,387,224,0.621,0.739,0.674,0.720,811,847,46,0.489,0.946,0.645,0.200
4,4,calendario_2025,3,0.050,8.876e-04,0.015,0,60,3,0.000,0.000,0.000,0.006,0,203,3,0.000,0.000,0.000,0.225
5,4,temporada_2022,431,0.112,1.275e-01,0.084,151,134,280,0.530,0.350,0.422,0.528,256,292,175,0.467,0.594,0.523,0.275
6,4,temporada_2023,1039,0.174,3.074e-01,0.127,495,57,544,0.897,0.476,0.622,0.834,833,217,206,0.793,0.802,0.798,0.175
7,4,temporada_2024,857,0.302,2.536e-01,0.133,590,418,267,0.585,0.688,0.633,0.653,793,922,64,0.462,0.925,0.617,0.175


In [8]:
pd.concat(fiabilidad, names=["horizonte", "bin"])

n  prob_media  frecuencia_observada
horizonte bin                                               
2         [0.0, 0.1)  6151       0.012                 0.033
          [0.1, 0.2)   538       0.144                 0.197
          [0.2, 0.3)   498       0.249                 0.345
          [0.3, 0.4)   419       0.351                 0.394
          [0.4, 0.5)   478       0.451                 0.496
          [0.5, 0.6)   478       0.550                 0.529
          [0.6, 0.7)   512       0.647                 0.676
          [0.7, 0.8)   510       0.751                 0.729
          [0.8, 0.9)   377       0.844                 0.836
          [0.9, 1.0)   179       0.934                 0.888
4         [0.0, 0.1)  6103       0.014                 0.043
          [0.1, 0.2)   709       0.148                 0.248
          [0.2, 0.3)   523       0.251                 0.342
          [0.3, 0.4)   451       0.351                 0.428
          [0.4, 0.5)   509       0.451                 0.546
          [0.5, 0.6)   538       0.551                 0.599
          [0.6, 0.7)   513       0.652                 0.659
          [0.7, 0.8)   369       0.749                 0.675
          [0.8, 0.9)   287       0.850                 0.739
          [0.9, 1.0)   138       0.929                 0.833

### 1.4 Detección de episodios

Episodio: semanas `brote` consecutivas, uniendo huecos de hasta 2 semanas. Antelación = semanas entre la emisión de la primera alerta pertinente y el inicio del episodio (positiva: avisó antes). «Inicio limpio»: sin `brote` en las 8 semanas previas del distrito, para que una alerta temprana no sea arrastre del episodio anterior.

In [9]:
antelacion = d.antelacion_episodios(diag, etiquetas, ALERTAS)
agregado = antelacion.assign(bloque="2022_2024")
antelacion_todos = d.resumen_antelacion(agregado, ALERTAS)
antelacion_limpios = d.resumen_antelacion(agregado, ALERTAS, d.SIN_BROTE_PREVIO)
antelacion_limpios_temporada = d.resumen_antelacion(antelacion, ALERTAS, d.SIN_BROTE_PREVIO)
d.guardar_figura(d.figura_antelacion(antelacion[antelacion.semanas_previas_sin_brote >= d.SIN_BROTE_PREVIO], ALERTAS),
                 FIGURAS_ARQUITECTURA, "antelacion_inicios_limpios")
pd.concat({"todos": antelacion_todos, "inicios_limpios": antelacion_limpios}, names=["episodios"])

horizonte     bloque                alerta  episodios  \
episodios                                                                  
todos           0          2  2022_2024  alerta_cls_calibrada        206   
                1          2  2022_2024        alerta_cls_050        206   
                2          2  2022_2024      alerta_regresion        206   
                3          2  2022_2024   alerta_persistencia        206   
                4          4  2022_2024  alerta_cls_calibrada        206   
                5          4  2022_2024        alerta_cls_050        206   
                6          4  2022_2024      alerta_regresion        206   
                7          4  2022_2024   alerta_persistencia        206   
inicios_limpios 0          2  2022_2024  alerta_cls_calibrada        118   
                1          2  2022_2024        alerta_cls_050        118   
                2          2  2022_2024      alerta_regresion        118   
                3          2  2022_2024   alerta_persistencia        118   
                4          4  2022_2024  alerta_cls_calibrada        118   
                5          4  2022_2024        alerta_cls_050        118   
                6          4  2022_2024      alerta_regresion        118   
                7          4  2022_2024   alerta_persistencia        118   

                   detectados  antes_del_inicio  antelacion_2_o_mas  \
episodios                                                             
todos           0       0.704             0.451               0.427   
                1       0.505             0.175               0.165   
                2       0.558             0.175               0.160   
                3       0.680             0.282               0.277   
                4       0.694             0.490               0.481   
                5       0.456             0.189               0.184   
                6       0.573             0.233               0.228   
                7       0.714             0.427               0.413   
inicios_limpios 0       0.669             0.288               0.271   
                1       0.483             0.110               0.102   
                2       0.542             0.102               0.085   
                3       0.669             0.127               0.119   
                4       0.653             0.356               0.339   
                5       0.390             0.102               0.102   
                6       0.517             0.102               0.102   
                7       0.636             0.203               0.186   

                   antelacion_mediana  
episodios                              
todos           0                 4.0  
                1                -1.0  
                2                 0.0  
                3                 0.0  
                4                 5.0  
                5                 0.0  
                6                 0.0  
                7                 4.0  
inicios_limpios 0                 0.0  
                1                -1.0  
                2                 0.0  
                3                 0.0  
                4                 2.0  
                5                -1.0  
                6                 0.0  
                7                 0.0

In [10]:
antelacion_limpios_temporada

,horizonte,bloque,alerta,episodios,detectados,antes_del_inicio,antelacion_2_o_mas,antelacion_mediana
0,2,temporada_2022,alerta_cls_calibrada,38,0.500,0.211,0.211,0.0
1,2,temporada_2022,alerta_cls_050,38,0.316,0.053,0.053,-1.0
2,2,temporada_2022,alerta_regresion,38,0.368,0.079,0.053,-1.0
3,2,temporada_2022,alerta_persistencia,38,0.553,0.132,0.105,0.0
4,2,temporada_2023,alerta_cls_calibrada,57,0.737,0.281,0.281,0.0
5,2,temporada_2023,alerta_cls_050,57,0.561,0.140,0.123,-2.0
6,2,temporada_2023,alerta_regresion,57,0.632,0.105,0.088,0.0
7,2,temporada_2023,alerta_persistencia,57,0.754,0.123,0.123,0.0
8,2,temporada_2024,alerta_cls_calibrada,23,0.783,0.435,0.348,1.0
9,2,temporada_2024,alerta_cls_050,23,0.565,0.130,0.130,-1.0


### 1.5 Concentración del error (2022–2024)

Los cinco distritos son los de más casos observados en esas tres temporadas.

In [11]:
concentracion = d.concentracion_error(diag)
concentracion

,horizonte,pronostico,grupo,fraccion_filas,fraccion_casos,fraccion_error_absoluto,fraccion_error_cuadratico
0,2,modelo,temporada_2023,0.333,0.605,0.585,0.834
1,2,modelo,top5_distritos,0.077,0.483,0.470,0.835
2,2,modelo,ambos,0.026,0.309,0.316,0.739
3,2,persistencia,temporada_2023,0.333,0.605,0.610,0.896
4,2,persistencia,top5_distritos,0.077,0.483,0.443,0.810
5,2,persistencia,ambos,0.026,0.309,0.308,0.754
6,4,modelo,temporada_2023,0.333,0.605,0.533,0.716
7,4,modelo,top5_distritos,0.077,0.483,0.463,0.760
8,4,modelo,ambos,0.026,0.309,0.281,0.604
9,4,persistencia,temporada_2023,0.333,0.605,0.623,0.899


In [12]:
ruta = d.guardar_seccion_metricas(METRICAS_ARQUITECTURA, "fase1_diagnostico", {
    "fuente": {"predicciones": "models/experimentos/validacion_temporal_compacta/predicciones_por_bloque.csv",
               "regresion": d.VARIANTE_REGRESION, "clasificacion": d.VARIANTE_CLASIFICACION,
               "bloques_diagnostico": list(d.BLOQUES_DIAGNOSTICO), "semilla": reporte["parametros"]["random_state"],
               "device": reporte["parametros"]["device"]},
    "definiciones": {"fase_curva": "curva regional del bloque: valle <10 % del máximo, pico >=60 %",
                     "tercil_distrito": "casos acumulados antes de la temporada 2021",
                     "episodio": "brote consecutivo uniendo huecos <=2 semanas",
                     "antelacion": "h - (w - t0); alertas con objetivo desde t0-2",
                     "inicio_limpio_semanas_sin_brote": d.SIN_BROTE_PREVIO,
                     "adyacencia_falso_positivo": "brote del mismo distrito a ±2 semanas"},
    "errores_por_temporada": d.a_json(errores_temporada),
    "errores_por_tercil": d.a_json(errores_tercil),
    "errores_por_fase": d.a_json(errores_fase.astype({"fase": str})),
    "errores_por_nivel": d.a_json(errores_nivel.astype({"nivel": str})),
    "falsos_positivos": d.a_json(fp), "falsos_positivos_total": d.a_json(fp_total),
    "probabilidad_por_temporada": d.a_json(probabilidad),
    "fiabilidad_2022_2024": {str(h): d.a_json(t.reset_index(names="bin")) for h, t in fiabilidad.items()},
    "antelacion_todos": d.a_json(antelacion_todos),
    "antelacion_inicios_limpios": d.a_json(antelacion_limpios),
    "antelacion_inicios_limpios_por_temporada": d.a_json(antelacion_limpios_temporada),
    "concentracion": d.a_json(concentracion),
})
print(ruta.relative_to(ROOT))

docs\modeling\metricas\arquitectura_horizontes.json


## Fase 2: protocolo de evaluación

El protocolo está escrito en `docs/modeling/arquitectura_horizontes.md` (sección «Protocolo») e implementado en `src/modeling/protocolo_arquitectura.py`. Aquí se aplica a la arquitectura actual (E1: XGBoost directo por horizonte, h=2/4) con las cinco semillas de la fase 0, para fijar la **línea base bajo el protocolo** y los **rangos de empate** antes de comparar arquitecturas. Ninguna arquitectura nueva se evalúa en esta fase.

In [13]:
from src.modeling import protocolo_arquitectura as pa
from src.utils.paths import EXPERIMENTOS_ARQUITECTURA

evaluaciones = {}
for semilla in pa.SEMILLAS:
    archivo = EXPERIMENTOS_ARQUITECTURA / "fase0_semillas" / f"semilla_{semilla}" / "predicciones_por_bloque.csv"
    evaluaciones[semilla] = pa.evaluar(pa.desde_compacta(pd.read_csv(archivo, dtype={"ubigeo": "string"})), etiquetas)
e1 = evaluaciones[17]
e1["calibracion"]

,horizonte,bloque,corte,metodo,filas_calibracion,positivos_calibracion,etiquetas_no_cerradas,ultima_etiqueta_usada
0,2,temporada_2021,2020-08-15,platt,0,0,0,NaT
1,2,temporada_2022,2021-08-21,platt,3380,84,65,2021-08-21
2,2,temporada_2023,2022-08-20,platt,6760,519,65,2022-08-20
3,2,temporada_2024,2023-08-19,platt,10140,1524,65,2023-08-19
4,2,calendario_2025,2024-12-21,platt,13585,2417,0,2024-08-24
5,4,temporada_2021,2020-08-01,platt,0,0,0,NaT
6,4,temporada_2022,2021-08-07,platt,3250,71,195,2021-08-07
7,4,temporada_2023,2022-08-06,platt,6630,515,195,2022-08-06
8,4,temporada_2024,2023-08-05,platt,10010,1454,195,2023-08-05
9,4,calendario_2025,2024-12-07,platt,13585,2417,0,2024-08-24


### Línea base E1 (semilla 17) con la regla de las HU

Alerta con probabilidad calibrada ≥ 0,50. Las columnas `diagnostico_*` usan el umbral que maximiza F1 en la calibración previa: **solo diagnóstico**, no decide. 2024 es contraste ya inspeccionado; 2025 solo vale para conteos.

In [14]:
columnas = ["horizonte", "bloque", "positivos", "vp_050", "fp_050", "fn_050", "precision_050", "recall_050",
            "f1_050", "auprc", "brier", "brier_sin_calibrar", "prob_media_calibrada", "prevalencia",
            "f1_regresion", "f1_persistencia", "mae", "mae_persistencia", "razon_mae", "rmse",
            "rmse_persistencia", "razon_rmse", "diagnostico_umbral_f1", "diagnostico_f1"]
e1["metricas"][columnas]

,horizonte,bloque,positivos,vp_050,fp_050,fn_050,precision_050,recall_050,f1_050,auprc,...,f1_regresion,f1_persistencia,mae,mae_persistencia,razon_mae,rmse,rmse_persistencia,razon_rmse,diagnostico_umbral_f1,diagnostico_f1
0,2,calendario_2025,3,0.0,52.0,3.0,0.000,0.000,0.000,0.010,...,0.000,0.000,0.301,0.285,1.055,1.022,0.981,1.042,0.325,0.015
1,2,temporada_2021,90,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.519,0.610,0.515,0.507,1.016,3.243,3.143,1.032,NaN,NaN
2,2,temporada_2022,431,215.0,208.0,216.0,0.508,0.499,0.504,0.558,...,0.679,0.686,1.944,1.959,0.992,8.037,6.036,1.332,0.275,0.554
3,2,temporada_2023,1039,595.0,67.0,444.0,0.899,0.573,0.700,0.881,...,0.834,0.861,10.687,10.837,0.986,56.377,49.474,1.140,0.250,0.834
4,2,temporada_2024,857,753.0,622.0,104.0,0.548,0.879,0.675,0.720,...,0.802,0.798,5.639,4.956,1.138,23.848,15.757,1.514,0.350,0.644
5,4,calendario_2025,3,0.0,74.0,3.0,0.000,0.000,0.000,0.006,...,0.000,0.000,0.433,0.324,1.334,1.661,1.168,1.422,0.325,0.000
6,4,temporada_2021,90,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.317,0.479,0.679,0.674,1.007,4.728,4.612,1.025,NaN,NaN
7,4,temporada_2022,431,142.0,124.0,289.0,0.534,0.329,0.407,0.528,...,0.613,0.638,3.074,2.846,1.080,16.463,8.536,1.929,0.300,0.520
8,4,temporada_2023,1039,428.0,51.0,611.0,0.894,0.412,0.564,0.834,...,0.772,0.813,14.249,17.680,0.806,69.383,80.185,0.865,0.225,0.799
9,4,temporada_2024,857,718.0,689.0,139.0,0.510,0.838,0.634,0.653,...,0.758,0.733,9.401,7.848,1.198,40.443,25.455,1.589,0.375,0.620


In [15]:
# Niveles de riesgo (cortes 0,25 / 0,50 / 0,75) y tasa observada de brote en cada nivel.
e1["metricas"][["horizonte", "bloque"] + [c for c in e1["metricas"] if c.startswith(("n_", "tasa_brote_"))]]

,horizonte,bloque,n_bajo,tasa_brote_bajo,n_medio,tasa_brote_medio,n_alto,tasa_brote_alto,n_muy_alto,tasa_brote_muy_alto
0,2,calendario_2025,3188.0,3.137e-04,140.0,0.014,50.0,0.000,2.0,0.000
1,2,temporada_2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,temporada_2022,2694.0,4.380e-02,263.0,0.373,265.0,0.426,158.0,0.646
3,2,temporada_2023,2282.0,6.486e-02,436.0,0.679,547.0,0.883,115.0,0.974
4,2,temporada_2024,1598.0,2.190e-02,407.0,0.170,696.0,0.382,679.0,0.717
5,4,calendario_2025,3058.0,3.270e-04,248.0,0.008,70.0,0.000,4.0,0.000
6,4,temporada_2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,4,temporada_2022,2760.0,5.471e-02,354.0,0.390,230.0,0.470,36.0,0.944
8,4,temporada_2023,2378.0,9.462e-02,523.0,0.738,436.0,0.888,43.0,0.953
9,4,temporada_2024,1486.0,2.557e-02,487.0,0.207,715.0,0.359,692.0,0.666


In [16]:
d.guardar_figura(d.figura_fiabilidad(e1["datos"][e1["datos"].probabilidad_calibrada.notna()]
                                     .assign(probabilidad_brote=lambda t: t.probabilidad_calibrada)),
                 FIGURAS_ARQUITECTURA, "fiabilidad_calibrada_e1")
e1["fiabilidad"].pivot_table(index=["horizonte", "bin"], columns="bloque", values="frecuencia_observada")

bloque                calendario_2025  seleccion_2022_2023  temporada_2022  \
horizonte bin                                                                
2         [0.0, 0.1)        3.500e-04                0.027           0.025   
          [0.1, 0.2)        0.000e+00                0.214           0.168   
          [0.2, 0.3)        9.709e-03                0.402           0.306   
          [0.3, 0.4)        1.923e-02                0.496           0.359   
          [0.4, 0.5)        0.000e+00                0.679           0.429   
          [0.5, 0.6)        0.000e+00                0.687           0.305   
          [0.6, 0.7)        0.000e+00                0.749           0.495   
          [0.7, 0.8)        0.000e+00                0.804           0.581   
          [0.8, 0.9)        0.000e+00                0.745           0.662   
          [0.9, 1.0)              NaN                1.000           1.000   
4         [0.0, 0.1)        0.000e+00                0.037           0.032   
          [0.1, 0.2)        0.000e+00                0.269           0.214   
          [0.2, 0.3)        1.478e-02                0.392           0.304   
          [0.3, 0.4)        0.000e+00                0.518           0.338   
          [0.4, 0.5)        0.000e+00                0.752           0.505   
          [0.5, 0.6)        0.000e+00                0.778           0.370   
          [0.6, 0.7)        0.000e+00                0.733           0.533   
          [0.7, 0.8)        0.000e+00                0.704           0.673   
          [0.8, 0.9)        0.000e+00                0.976           1.000   
          [0.9, 1.0)              NaN                  NaN             NaN   

bloque                temporada_2023  temporada_2024  
horizonte bin                                         
2         [0.0, 0.1)           0.030           0.018  
          [0.1, 0.2)           0.253           0.052  
          [0.2, 0.3)           0.487           0.063  
          [0.3, 0.4)           0.596           0.140  
          [0.4, 0.5)           0.788           0.214  
          [0.5, 0.6)           0.878           0.341  
          [0.6, 0.7)           0.866           0.392  
          [0.7, 0.8)           0.948           0.492  
          [0.8, 0.9)           0.967           0.685  
          [0.9, 1.0)             NaN           0.842  
4         [0.0, 0.1)           0.044           0.019  
          [0.1, 0.2)           0.313           0.050  
          [0.2, 0.3)           0.467           0.099  
          [0.3, 0.4)           0.676           0.211  
          [0.4, 0.5)           0.861           0.242  
          [0.5, 0.6)           0.916           0.314  
          [0.6, 0.7)           0.889           0.357  
          [0.7, 0.8)           0.739           0.504  
          [0.8, 0.9)           0.960           0.663  
          [0.9, 1.0)             NaN           0.764

In [17]:
# Antelación sobre inicios limpios: fracción avisada ≥ 2 semanas antes, IC 95 % bootstrap por episodio.
e1["antelacion_ic"]

,horizonte,bloque,alerta,episodios,avisados_2_o_mas,ic95_inferior,ic95_superior
0,2,temporada_2022,alerta_050,38,0.105,0.026,0.211
1,2,temporada_2023,alerta_050,57,0.105,0.035,0.193
2,2,temporada_2024,alerta_050,23,0.174,0.043,0.348
3,2,seleccion_2022_2023,alerta_050,95,0.105,0.053,0.168
4,4,temporada_2022,alerta_050,38,0.053,0.000,0.132
5,4,temporada_2023,alerta_050,57,0.105,0.035,0.193
6,4,temporada_2024,alerta_050,23,0.130,0.000,0.261
7,4,seleccion_2022_2023,alerta_050,95,0.084,0.032,0.147
8,2,temporada_2022,alerta_persistencia,38,0.105,0.026,0.211
9,2,temporada_2023,alerta_persistencia,57,0.123,0.053,0.211


In [18]:
e1["errores_por_fase"].query("bloque in @pa.BLOQUES_SELECCION")

,horizonte,bloque,fase,n,casos_medios,mae_modelo,mae_persistencia,razon_mae,rmse_modelo,rmse_persistencia,razon_rmse,sesgo_modelo,sesgo_persistencia,suma_predicha_sobre_real
8,2,temporada_2022,valle,1625.0,0.892,0.558,0.699,0.798,1.819,2.168,0.839,-0.154,0.187,0.827
9,2,temporada_2022,subida,715.0,4.116,2.883,2.252,1.280,12.724,7.024,1.812,-0.118,-1.250,0.971
10,2,temporada_2022,pico,585.0,10.954,4.867,4.265,1.141,12.356,9.275,1.332,1.110,-0.771,1.101
11,2,temporada_2022,bajada,455.0,4.200,1.660,3.035,0.547,4.161,8.106,0.513,-0.013,2.468,0.997
12,2,temporada_2023,valle,1950.0,2.594,1.293,1.293,1.000,4.637,4.531,1.023,-0.653,-0.502,0.748
13,2,temporada_2023,subida,455.0,30.833,12.332,13.040,0.946,31.975,35.599,0.898,-4.685,-10.125,0.848
14,2,temporada_2023,pico,390.0,99.572,54.639,45.495,1.201,156.893,123.321,1.272,-51.785,-12.587,0.480
15,2,temporada_2023,bajada,585.0,28.803,11.420,17.831,0.640,32.965,54.308,0.607,-7.510,15.308,0.739
28,4,temporada_2022,valle,1625.0,0.892,0.665,0.982,0.677,2.219,3.279,0.677,-0.327,0.420,0.633
29,4,temporada_2022,subida,715.0,4.116,3.286,2.937,1.119,12.028,9.024,1.333,-1.222,-2.308,0.703


### Rangos de empate entre semillas (E1, h=2 y h=4)

Máximo − mínimo de cada métrica entre las cinco semillas. Una diferencia entre arquitecturas menor que el mayor de los rangos de ambas es empate. h=3 aún no tiene semillas: usa el mayor rango de h=2/h=4 hasta que se mida.

In [19]:
metricas_semillas = [evaluaciones[s]["metricas"] for s in pa.SEMILLAS]
rangos_e1 = pa.rangos_semillas(metricas_semillas)
mediana_e1 = pa.mediana_semillas(metricas_semillas)
antelacion_semillas = pd.concat([evaluaciones[s]["antelacion_ic"].assign(semilla=s) for s in pa.SEMILLAS])
rango_antelacion = (antelacion_semillas.query("alerta == 'alerta_050'")
                    .groupby(["horizonte", "bloque"]).avisados_2_o_mas.agg(["min", "max"]).reset_index())
control = pa.decidir(evaluaciones[29]["metricas"], e1["metricas"], rangos_e1,
                     pa.diferencia_antelacion(evaluaciones[29]["antelacion"], e1["antelacion"], "alerta_050"))
display(rangos_e1, rango_antelacion)
control  # control de la regla: otra semilla de la misma arquitectura debe empatar

,horizonte,bloque,f1_050,auprc,brier,razon_mae,rmse,mae,recall_050,precision_050
0,2,calendario_2025,0.000,6.078e-04,5.318e-04,0.014,0.024,0.004,0.000,0.000
1,2,temporada_2021,NaN,NaN,NaN,0.024,0.109,0.012,NaN,NaN
2,2,temporada_2022,0.021,2.171e-02,1.753e-03,0.015,0.171,0.030,0.026,0.028
3,2,temporada_2023,0.014,4.559e-03,1.790e-03,0.023,0.912,0.249,0.016,0.008
4,2,temporada_2024,0.002,1.142e-02,1.175e-03,0.056,1.248,0.278,0.007,0.001
5,4,calendario_2025,0.000,9.719e-04,2.686e-04,0.014,0.022,0.004,0.000,0.000
6,4,temporada_2021,NaN,NaN,NaN,0.017,0.169,0.012,NaN,NaN
7,4,temporada_2022,0.013,1.226e-02,1.053e-03,0.060,2.219,0.172,0.016,0.016
8,4,temporada_2023,0.024,4.174e-03,2.312e-03,0.012,0.818,0.204,0.024,0.005
9,4,temporada_2024,0.003,4.898e-03,9.361e-04,0.051,3.098,0.401,0.012,0.002


,horizonte,bloque,min,max
0,2,seleccion_2022_2023,0.105,0.116
1,2,temporada_2022,0.079,0.105
2,2,temporada_2023,0.105,0.123
3,2,temporada_2024,0.174,0.174
4,4,seleccion_2022_2023,0.084,0.095
5,4,temporada_2022,0.053,0.053
6,4,temporada_2023,0.105,0.123
7,4,temporada_2024,0.130,0.130


,horizonte,veredicto,mejora_f1_050,empeora_f1_050,mejora_antelacion,empeora_antelacion,vetos,vetos_contra_referencia,umbral_f1_050_temporada_2022,umbral_f1_050_temporada_2023,...,umbral_razon_mae_temporada_2022,umbral_razon_mae_temporada_2023,delta_f1_050_temporada_2022,delta_f1_050_temporada_2023,delta_auprc_temporada_2022,delta_auprc_temporada_2023,delta_brier_temporada_2022,delta_brier_temporada_2023,delta_razon_mae_temporada_2022,delta_razon_mae_temporada_2023
0,2,empate,False,False,False,False,None,None,0.021,0.014,...,0.015,0.023,-0.003,-0.008,-0.001,0.002,-3.054e-04,-1.415e-04,-0.010,-6.582e-03
1,4,empate,False,False,False,False,None,None,0.013,0.024,...,0.060,0.012,-0.013,0.005,-0.007,0.002,4.842e-04,-9.038e-04,0.013,4.542e-04


In [20]:
ruta = d.guardar_seccion_metricas(METRICAS_ARQUITECTURA, "fase2_protocolo", {
    "documento": "docs/modeling/arquitectura_horizontes.md#protocolo",
    "modulo": "src/modeling/protocolo_arquitectura.py",
    "bloques": {"calibracion_inicial": pa.BLOQUE_CALIBRACION_INICIAL, "seleccion": list(pa.BLOQUES_SELECCION),
                "contraste_no_decisorio": pa.BLOQUE_CONTRASTE, "sensibilidad_conteos": pa.BLOQUE_SENSIBILIDAD_CONTEOS,
                "test_independiente": "semanas de 2026 publicadas tras congelar la arquitectura (HU0019)"},
    "regla_alerta": {"umbral": pa.UMBRAL_ALERTA, "cortes_riesgo": pa.CORTES_RIESGO,
                     "calibracion": pa.METODO_CALIBRACION, "umbral_f1": "solo diagnóstico"},
    "antelacion": {"minima_semanas": pa.ANTELACION_MINIMA, "inicio_limpio_semanas_sin_brote": d.SIN_BROTE_PREVIO,
                   "hueco_max_episodio": d.VENTANA_ADYACENCIA, "bootstrap": pa.N_BOOTSTRAP,
                   "semilla_bootstrap": pa.SEMILLA_BOOTSTRAP},
    "semillas": list(pa.SEMILLAS), "device": reporte["parametros"]["device"],
    "calibracion_e1_semilla_17": d.a_json(e1["calibracion"]),
    "linea_base_e1_semilla_17": d.a_json(e1["metricas"]),
    "linea_base_e1_mediana_semillas": d.a_json(mediana_e1),
    "antelacion_e1_semilla_17": d.a_json(e1["antelacion_ic"]),
    "errores_por_fase_e1_semilla_17": d.a_json(e1["errores_por_fase"].astype({"fase": str})),
    "fiabilidad_calibrada_e1_semilla_17": d.a_json(e1["fiabilidad"]),
    "rangos_empate_e1": d.a_json(rangos_e1),
    "rango_antelacion_e1": d.a_json(rango_antelacion),
    "control_regla_semilla_29_vs_17": d.a_json(control),
})
print(ruta.relative_to(ROOT))

docs\modeling\metricas\arquitectura_horizontes.json


## Fase 3: estrategia multi-horizonte

Matrices de h = 1 a 6 construidas **en memoria** con `features.construir_gold` (el mismo código que genera gold h=2/h=4) y auditadas con `train.preparar_gold`; nada se escribe en `data/`. Variables congeladas: `base_6` para regresión y `base_6_poblacion_2017` para clasificación. Cada estrategia corre con las 5 semillas del protocolo y se evalúa con `protocolo_arquitectura.evaluar` (Platt cronológico, p ≥ 0,50).

In [21]:
import warnings
from src.eda.carga import cargar_integrado
from src.modeling import estrategias_horizonte as eh
from src.modeling.etiqueta_brote import ConfiguracionBroteEstacional
from src.modeling.features import construir_etiquetas
from src.modeling.seguimiento_mlflow import registrar_o_avisar
from src.utils.paths import GOLD_MANIFEST

warnings.filterwarnings("ignore", message=".*mismatched devices.*")
panel = cargar_integrado(SILVER_INTEGRADO)
regla = json.loads(GOLD_MANIFEST.read_text(encoding="utf-8"))["regla_brote"]
etiquetas_panel = construir_etiquetas(panel, ConfiguracionBroteEstacional(
    regla["multiplicador"], regla["anios_previos"], regla["minimo_casos"]))
serie = panel[["ubigeo", "anio", "semana", "semana_inicio", "casos_Dengue"]].merge(
    etiquetas_panel[["ubigeo", "anio", "semana", "brote"]], on=["ubigeo", "anio", "semana"])
serie["semana_inicio"] = pd.to_datetime(serie.semana_inicio)
matrices = {h: eh.matriz_horizonte(panel, etiquetas_panel, h) for h in eh.HORIZONTES_CURVA}
auditoria = pd.DataFrame([{"horizonte": h, "filas": len(m), "causal": bool((m.semana_inicio > m.origen_cierre).all()),
                           "distancia_origen_dias": sorted((m.semana_inicio - m.origen_cierre).dt.days.unique())}
                          for h, m in matrices.items()])
auditoria

,horizonte,filas,causal,distancia_origen_dias
0,1,30225,True,[1]
1,2,30160,True,[8]
2,3,30095,True,[15]
3,4,30030,True,[22]
4,5,29965,True,[29]
5,6,29900,True,[36]


In [22]:
def _calcular_fase3_entrenamiento():
    # Entrenamiento: E1 para h=1..6 y E2/E3 para h=2..4, cinco semillas cada una.
    predicciones = {"E1": {}, "E2": {}, "E3": {}}
    evaluaciones = {"E1": {}, "E2": {}, "E3": {}}
    for semilla in pa.SEMILLAS:
        predicciones["E1"][semilla] = eh.predecir_e1(matrices, semilla)
        predicciones["E2"][semilla] = eh.predecir_e2(matrices, semilla)
        predicciones["E3"][semilla] = eh.predecir_e3(matrices, semilla)
        for estrategia in evaluaciones:
            evaluaciones[estrategia][semilla] = pa.evaluar(predicciones[estrategia][semilla], serie)
    return predicciones, evaluaciones

predicciones, evaluaciones = cache.cargar_o_calcular("fase3_entrenamiento", _calcular_fase3_entrenamiento, huella=HUELLA, forzar=FORZAR_REENTRENO)
# Control: E1 en memoria reproduce el experimento 30 (h=2 y h=4, semilla 17).
compacta = pd.read_csv(PREDICCIONES_COMPACTAS, dtype={"ubigeo": "string"})
control_e1 = []
for h, variante, columna in [(2, "base_6", "casos_predichos"), (4, "base_6", "casos_predichos"),
                             (2, "base_6_poblacion_2017", "probabilidad_brote"),
                             (4, "base_6_poblacion_2017", "probabilidad_brote")]:
    cruce = compacta[compacta.horizonte.eq(h) & compacta.variante.eq(variante)].merge(
        predicciones["E1"][17], on=["ubigeo", "anio", "semana", "bloque", "horizonte"])
    control_e1.append({"horizonte": h, "columna": columna, "filas": len(cruce),
                       "diferencia_maxima": float((cruce[f"{columna}_x"] - cruce[f"{columna}_y"]).abs().max())})
pd.DataFrame(control_e1)

caché: 'fase3_entrenamiento' recalculada (forzada) (huella 0e2c4bac8c6e)


,horizonte,columna,filas,diferencia_maxima
0,2,casos_predichos,16965,2.637e-05
1,4,casos_predichos,16965,2.549e-05
2,2,probabilidad_brote,16965,2.980e-08
3,4,probabilidad_brote,16965,2.979e-08


### 3.1 Curva de habilidad según el horizonte (E1, semilla 17)

F1 de alerta con p ≥ 0,50 calibrada para el XGBoost y con la regla de casos para la persistencia (casos `t−h`) y el ingenuo estacional (casos `t−52`). MAE relativo a la persistencia: < 1 es mejor; la línea de 0,85 es el criterio de la HU0007-4.

In [23]:
columnas_curva = ["horizonte", "bloque", "f1_050", "precision_050", "recall_050", "auprc", "brier",
                  "f1_regresion", "f1_persistencia", "f1_estacional", "auprc_persistencia", "auprc_estacional",
                  "razon_mae", "razon_rmse", "razon_mae_estacional"]
curva = evaluaciones["E1"][17]["metricas"].query("bloque in @pa.BLOQUES_SELECCION or bloque == @pa.BLOQUE_CONTRASTE")
d.guardar_figura(eh.figura_curva_habilidad(curva[curva.bloque.isin(pa.BLOQUES_SELECCION)]),
                 FIGURAS_ARQUITECTURA, "curva_habilidad_horizonte")
curva[columnas_curva].sort_values(["bloque", "horizonte"])

,horizonte,bloque,f1_050,precision_050,recall_050,auprc,brier,f1_regresion,f1_persistencia,f1_estacional,auprc_persistencia,auprc_estacional,razon_mae,razon_rmse,razon_mae_estacional
2,1,temporada_2022,0.572,0.544,0.603,0.605,0.079,0.692,0.734,0.043,0.754,0.118,1.025,1.135,2.471
7,2,temporada_2022,0.504,0.508,0.499,0.558,0.081,0.679,0.686,0.043,0.676,0.118,0.992,1.332,1.870
12,3,temporada_2022,0.454,0.549,0.387,0.542,0.081,0.650,0.679,0.043,0.640,0.118,1.009,1.572,1.535
17,4,temporada_2022,0.407,0.534,0.329,0.528,0.082,0.613,0.638,0.043,0.576,0.118,1.080,1.929,1.287
22,5,temporada_2022,0.381,0.531,0.297,0.506,0.085,0.576,0.581,0.043,0.506,0.118,1.167,2.511,1.118
27,6,temporada_2022,0.363,0.506,0.283,0.485,0.089,0.553,0.569,0.043,0.480,0.118,1.262,2.676,1.006
3,1,temporada_2023,0.753,0.905,0.644,0.897,0.093,0.869,0.877,0.452,0.921,0.457,1.144,1.404,2.918
8,2,temporada_2023,0.700,0.899,0.573,0.881,0.105,0.834,0.861,0.452,0.899,0.457,0.986,1.140,1.873
13,3,temporada_2023,0.651,0.903,0.509,0.853,0.118,0.792,0.829,0.452,0.868,0.457,0.871,0.955,1.386
18,4,temporada_2023,0.564,0.894,0.412,0.834,0.127,0.772,0.813,0.452,0.844,0.457,0.806,0.865,1.148


In [24]:
antelacion_curva = evaluaciones["E1"][17]["antelacion_ic"].query("bloque == 'seleccion_2022_2023'")
antelacion_curva.pivot_table(index="horizonte", columns="alerta", values="avisados_2_o_mas")

alerta,alerta_050,alerta_persistencia,alerta_regresion
horizonte,,,
1,0.105,0.042,0.032
2,0.105,0.116,0.074
3,0.095,0.137,0.095
4,0.084,0.211,0.095
5,0.074,0.189,0.095
6,0.074,0.179,0.105


### 3.2 Horizonte nominal (decidido)

`h` es la distancia entre el último dato disponible (`t−h`) y la semana objetivo. La demora de publicación de MINSA es una limitación aceptada: no se modela ni se simula.

### 3.3 Estrategias E1, E2 y E3 (h = 2, 3, 4)

Medianas de 5 semillas en las temporadas de selección y el contraste 2024 (no decide). E4 (recursiva) se evalúa al final solo con la curva 3.1.

In [25]:
evaluaciones_234 = {e: {s: eh.restringir_horizontes(r) for s, r in evaluaciones[e].items()} for e in evaluaciones}
medianas = pd.concat({e: pa.mediana_semillas([r["metricas"] for r in evaluaciones_234[e].values()])
                      for e in evaluaciones_234}, names=["estrategia"]).reset_index(level=0)
columnas_tabla = ["estrategia", "horizonte", "bloque", "precision_050", "recall_050", "f1_050", "auprc", "brier",
                  "f1_regresion", "f1_persistencia", "razon_mae", "razon_rmse"]
tabla_estrategias = medianas.query("bloque in @pa.BLOQUES_SELECCION or bloque == @pa.BLOQUE_CONTRASTE")[columnas_tabla]
tabla_estrategias.sort_values(["horizonte", "bloque", "estrategia"])

,estrategia,horizonte,bloque,precision_050,recall_050,f1_050,auprc,brier,f1_regresion,f1_persistencia,razon_mae,razon_rmse
2,E1,2,temporada_2022,0.520,0.503,0.511,0.563,0.081,0.679,0.686,0.987,1.332
2,E2,2,temporada_2022,0.531,0.455,0.491,0.533,0.082,0.638,0.686,1.113,1.640
2,E3,2,temporada_2022,0.525,0.485,0.505,0.546,0.082,0.668,0.686,1.004,1.413
3,E1,2,temporada_2023,0.895,0.564,0.693,0.881,0.105,0.832,0.861,0.986,1.140
3,E2,2,temporada_2023,0.897,0.539,0.674,0.867,0.110,0.828,0.861,1.050,1.170
3,E3,2,temporada_2023,0.897,0.574,0.700,0.878,0.105,0.831,0.861,1.001,1.160
4,E1,2,temporada_2024,0.547,0.882,0.675,0.720,0.140,0.802,0.798,1.151,1.514
4,E2,2,temporada_2024,0.550,0.866,0.673,0.713,0.140,0.791,0.798,1.303,1.805
4,E3,2,temporada_2024,0.552,0.872,0.676,0.710,0.139,0.800,0.798,1.220,1.657
7,E1,3,temporada_2022,0.549,0.390,0.455,0.542,0.080,0.652,0.679,1.009,1.572


In [26]:
rangos = pd.concat({e: pa.rangos_semillas([r["metricas"] for r in evaluaciones_234[e].values()])
                    for e in evaluaciones_234}, names=["estrategia"]).reset_index(level=0)
rangos.query("bloque in @pa.BLOQUES_SELECCION")[["estrategia", "horizonte", "bloque", "f1_050", "auprc", "brier", "razon_mae"]]

,estrategia,horizonte,bloque,f1_050,auprc,brier,razon_mae
2,E1,2,temporada_2022,0.021,0.022,1.753e-03,0.015
3,E1,2,temporada_2023,0.014,0.005,1.790e-03,0.023
7,E1,3,temporada_2022,0.005,0.017,9.332e-04,0.025
8,E1,3,temporada_2023,0.014,0.003,2.138e-03,0.009
12,E1,4,temporada_2022,0.013,0.012,1.053e-03,0.060
13,E1,4,temporada_2023,0.024,0.004,2.312e-03,0.012
2,E2,2,temporada_2022,0.009,0.014,9.328e-04,0.043
3,E2,2,temporada_2023,0.031,0.004,3.053e-03,0.023
7,E2,3,temporada_2022,0.008,0.010,5.103e-04,0.030
8,E2,3,temporada_2023,0.025,0.004,2.810e-03,0.013


In [27]:
antelacion_estrategias = pd.concat(
    {(e, s): r["antelacion_ic"].query("alerta == 'alerta_050'") for e in evaluaciones_234 for s, r in evaluaciones_234[e].items()},
    names=["estrategia", "semilla"]).reset_index(level=[0, 1])
antelacion_resumen = (antelacion_estrategias.groupby(["estrategia", "horizonte", "bloque"]).avisados_2_o_mas
                      .agg(mediana="median", minimo="min", maximo="max").reset_index())
antelacion_resumen.query("bloque == 'seleccion_2022_2023'")

,estrategia,horizonte,bloque,mediana,minimo,maximo
0,E1,2,seleccion_2022_2023,0.116,0.105,0.116
4,E1,3,seleccion_2022_2023,0.095,0.095,0.105
8,E1,4,seleccion_2022_2023,0.084,0.084,0.095
12,E2,2,seleccion_2022_2023,0.105,0.105,0.116
16,E2,3,seleccion_2022_2023,0.105,0.095,0.105
20,E2,4,seleccion_2022_2023,0.084,0.084,0.095
24,E3,2,seleccion_2022_2023,0.095,0.095,0.105
28,E3,3,seleccion_2022_2023,0.095,0.095,0.105
32,E3,4,seleccion_2022_2023,0.095,0.095,0.095


In [28]:
coherencias = {e: [eh.coherencia_horizontes(r["datos"], serie) for r in evaluaciones_234[e].values()] for e in evaluaciones_234}
coherencia = eh.resumen_coherencia(coherencias)
coherencia.query("bloque != 'calendario_2025'")

,estrategia,bloque,con_cambio,rango_con_cambio,cambio_sin_explicar,rango_cambio_sin_explicar,parpadeo,rango_parpadeo,alerta_h4_que_desaparece_en_h2,rango_alerta_h4_que_desaparece_en_h2
1,E1,temporada_2022,0.059,0.005,0.016,0.002,0.004,1.183e-03,0.004,2.071e-03
2,E1,temporada_2023,0.091,0.004,0.023,0.004,0.010,5.917e-04,0.013,3.550e-03
3,E1,temporada_2024,0.122,0.004,0.018,0.005,0.022,4.142e-03,0.055,4.142e-03
5,E2,temporada_2022,0.051,0.006,0.010,0.004,0.004,2.071e-03,0.004,1.183e-03
6,E2,temporada_2023,0.073,0.004,0.005,0.002,0.011,8.876e-04,0.020,1.479e-03
7,E2,temporada_2024,0.128,0.004,0.013,0.003,0.025,1.775e-03,0.067,1.479e-03
9,E3,temporada_2022,0.052,0.004,0.010,0.001,0.006,8.876e-04,0.003,8.876e-04
10,E3,temporada_2023,0.084,0.010,0.018,0.009,0.009,2.071e-03,0.010,2.663e-03
11,E3,temporada_2024,0.122,0.009,0.015,0.001,0.023,3.254e-03,0.064,4.438e-03


### Decisión preregistrada

Cada candidata frente a E1 por horizonte (umbral de empate: el mayor rango de semillas de las dos, h=3 con sus propias semillas). Una candidata se adopta si gana en algún horizonte y no pierde en ninguno; las empatadas pasan al desempate (coherencia y luego simplicidad).

In [29]:
veredictos = pd.concat([eh.decidir_frente_a_referencia(evaluaciones_234, c) for c in ("E2", "E3")], ignore_index=True)
resultado_adopcion = eh.adopcion(veredictos)
display(veredictos, resultado_adopcion)

,horizonte,veredicto,mejora_f1_050,empeora_f1_050,mejora_antelacion,empeora_antelacion,vetos,vetos_contra_referencia,umbral_f1_050_temporada_2022,umbral_f1_050_temporada_2023,...,delta_brier_temporada_2022,delta_brier_temporada_2023,delta_razon_mae_temporada_2022,delta_razon_mae_temporada_2023,candidata,referencia,antelacion_diferencia,antelacion_ic95_inferior,antelacion_ic95_superior,antelacion_signo_mediana_antelacion
0,2,empate,False,False,False,False,"auprc, razon_mae",None,0.021,0.031,...,8.132e-04,4.729e-03,0.126,0.064,E2,E1,0.011,0.0,0.032,-1.0
1,3,empate,False,False,False,False,NaN,None,0.008,0.025,...,1.494e-05,9.803e-05,-0.021,-0.003,E2,E1,0.000,0.0,0.000,1.0
2,4,empate,False,False,False,False,NaN,None,0.013,0.026,...,-5.599e-04,-4.241e-03,-0.193,-0.011,E2,E1,0.011,0.0,0.032,0.0
3,2,empate,False,False,False,False,None,None,0.021,0.014,...,8.092e-04,2.212e-04,0.017,0.015,E3,E1,0.000,0.0,0.000,-1.0
4,3,empate,False,False,False,False,None,None,0.013,0.015,...,1.350e-03,4.628e-04,0.011,0.001,E3,E1,0.011,0.0,0.032,0.0
5,4,empate,False,False,False,False,None,None,0.018,0.051,...,2.269e-03,-6.561e-04,0.015,-0.003,E3,E1,0.011,0.0,0.032,1.0


,candidata,gana_en,pierde_en,resultado
0,E2,None,None,empate
1,E3,None,None,empate


In [30]:
adoptables = resultado_adopcion.query("resultado == 'adoptable'").candidata.tolist()
empatadas = ["E1", *resultado_adopcion.query("resultado == 'empate'").candidata]
if len(adoptables) == 1:
    decision = {"elegida": adoptables[0], "criterio": "gana a E1 según la regla preregistrada"}
elif len(adoptables) > 1:
    decision = {"elegida": adoptables, "criterio": "varias adoptables: requiere comparación directa"}
else:
    decision = {**eh.desempatar(empatadas, coherencia), "empatadas": empatadas}
decision

{'elegida': ['E2', 'E3'],
 'criterio': 'empate sin resolver por simplicidad',
 'empatadas': ['E1', 'E2', 'E3']}

### E4 (recursiva)

Solo se evaluaría si la curva 3.1 mostrara que h=1 es mucho mejor que h=2.

In [31]:
e4 = curva[curva.horizonte.isin([1, 2]) & curva.bloque.isin(pa.BLOQUES_SELECCION)].pivot_table(
    index="bloque", columns="horizonte", values=["f1_050", "auprc", "razon_mae", "f1_persistencia"])
e4

auprc        f1_050        f1_persistencia        razon_mae  \
horizonte           1      2      1      2               1      2         1   
bloque                                                                        
temporada_2022  0.605  0.558  0.572  0.504           0.734  0.686     1.025   
temporada_2023  0.897  0.881  0.753  0.700           0.877  0.861     1.144   

                       
horizonte           2  
bloque                 
temporada_2022  0.992  
temporada_2023  0.986

In [32]:
destino = EXPERIMENTOS_ARQUITECTURA / "fase3"
destino.mkdir(parents=True, exist_ok=True)
for estrategia, por_semilla in predicciones.items():
    pd.concat(por_semilla.values(), ignore_index=True).to_csv(destino / f"predicciones_{estrategia}.csv.gz", index=False)
reporte_curva = eh.reporte_mlflow("fase3_curva_habilidad", {"E1": evaluaciones["E1"][17]["metricas"]}, (17,))
reporte_estrategias = eh.reporte_mlflow("fase3_estrategias_mediana_semillas",
                                        {e: medianas[medianas.estrategia.eq(e)] for e in evaluaciones_234}, pa.SEMILLAS)
registrar_o_avisar("arquitectura_horizontes_fase3_curva", reporte_curva, metricas_path=METRICAS_ARQUITECTURA,
                   salida_dir=destino, gold_dir=None)
registrar_o_avisar("arquitectura_horizontes_fase3_estrategias", reporte_estrategias, metricas_path=METRICAS_ARQUITECTURA,
                   salida_dir=destino, gold_dir=None)
rangos_h3 = rangos.query("estrategia == 'E1' and horizonte == 3")
ruta = d.guardar_seccion_metricas(METRICAS_ARQUITECTURA, "fase3_estrategias", {
    "matrices_en_memoria": d.a_json(auditoria.astype({"distancia_origen_dias": str})),
    "control_reproduccion_e1": control_e1,
    "curva_habilidad_e1_semilla_17": d.a_json(curva[columnas_curva]),
    "antelacion_curva_e1_semilla_17": d.a_json(antelacion_curva),
    "estrategias_mediana_semillas": d.a_json(tabla_estrategias),
    "rangos_semillas": d.a_json(rangos),
    "rangos_empate_h3_propios": d.a_json(rangos_h3),
    "antelacion_estrategias": d.a_json(antelacion_resumen),
    "coherencia": d.a_json(coherencia),
    "veredictos": d.a_json(veredictos), "adopcion": d.a_json(resultado_adopcion),
    "decision": {k: v for k, v in decision.items()},
    "modelos_por_estrategia": eh.MODELOS_POR_ESTRATEGIA,
    "e4_h1_frente_a_h2": d.a_json(e4.stack(0, future_stack=True).reset_index().rename(columns={"level_1": "metrica"})),
    "multi_output_tree": {"xgboost": reporte["version_xgboost"], "device": reporte["parametros"]["device"]},
})
print(ruta.relative_to(ROOT))

MLflow: run c65877d5bca44e8f84a245e0188de8fe en 'denguekay/arquitectura_horizontes_fase3_curva' (sqlite:///C:/Users/Rosa/RufoEsEternoElRegreso/Denguekay/mlflow.db)


MLflow: run ee7ab12f40e44a988e2dfb6130bab067 en 'denguekay/arquitectura_horizontes_fase3_estrategias' (sqlite:///C:/Users/Rosa/RufoEsEternoElRegreso/Denguekay/mlflow.db)
docs\modeling\metricas\arquitectura_horizontes.json


## Fase 4: formulación de la salida

Formulaciones S1–S5 (definidas en `src/modeling/salida_probabilistica.py`) sobre E3, la estrategia adoptada de forma provisional, y también sobre E1 para comparar al final el mejor par de cada una. Mismas variables, bloques, 5 semillas y protocolo (Platt cronológico salvo S5_iso, que usa isotónica). Requisito de S4: `tests/test_umbral_brote_sin_fuga.py` comprueba que `umbral_brote_casos` de la fila t no depende de casos posteriores a t−h (h=1 y h=6, datos sintéticos y panel real).

In [33]:
from src.modeling import salida_probabilistica as sp

compatibilidad = sp.compatibilidad_multi_salida()
compatibilidad

,capacidad,dispositivo,funciona,verificacion
0,count:poisson,cuda,True,media predicha − real por salida ≤ 0.0007
1,count:poisson,cpu,True,media predicha − real por salida ≤ 0.0007
2,reg:tweedie,cuda,True,"salida (2000, 3)"
3,reg:tweedie,cpu,True,"salida (2000, 3)"
4,base_margin con varias salidas (S3),cuda,True,"base_margin (n, 3) aplicado por columna; error..."
5,base_margin con varias salidas (S3),cpu,True,"base_margin (n, 3) aplicado por columna; error..."
6,importancia de variables (gain),cuda,True,feature_importances_ es agregada (una por vari...
7,importancia de variables (gain),cpu,True,feature_importances_ es agregada (una por vari...
8,"SHAP por horizonte, regresor",cuda,True,"pred_contribs (n, 3, p+1), aditivo por salida;..."
9,"SHAP por horizonte, regresor",cpu,True,"pred_contribs (n, 3, p+1), aditivo por salida;..."


In [34]:
pares = [sp.clave(e, f) for e in ("E1", "E3") for f in sp.FORMULACIONES]
def _calcular_fase4_entrenamiento():
    pares = [sp.clave(e, f) for e in ("E1", "E3") for f in sp.FORMULACIONES]
    evaluaciones_f, predicciones_f = {}, {}
    for par in pares:
        estrategia, formulacion = par.split("|")
        evaluaciones_f[par], predicciones_f[par] = {}, {}
        for semilla in pa.SEMILLAS:
            p = sp.PREDICTORES[estrategia](matrices, semilla, formulacion)
            predicciones_f[par][semilla] = p
            evaluaciones_f[par][semilla] = pa.evaluar(p, serie, sp.FORMULACIONES[formulacion]["calibracion"])
    return evaluaciones_f, predicciones_f

evaluaciones_f, predicciones_f = cache.cargar_o_calcular("fase4_entrenamiento", _calcular_fase4_entrenamiento, huella=HUELLA, forzar=FORZAR_REENTRENO)
medianas_f = pd.concat({par: pa.mediana_semillas([r["metricas"] for r in evaluaciones_f[par].values()]) for par in pares},
                       names=["par"]).reset_index(level=0)
rangos_f = pd.concat({par: pa.rangos_semillas([r["metricas"] for r in evaluaciones_f[par].values()]) for par in pares},
                     names=["par"]).reset_index(level=0)
columnas_f = ["par", "horizonte", "bloque", "precision_050", "recall_050", "f1_050", "auprc", "brier",
              "brier_sin_calibrar", "f1_regresion", "razon_mae", "razon_rmse"]
tabla_f = medianas_f.query("bloque in @pa.BLOQUES_SELECCION or bloque == @pa.BLOQUE_CONTRASTE")[columnas_f]
tabla_f.sort_values(["horizonte", "bloque", "par"])

caché: 'fase4_entrenamiento' recalculada (forzada) (huella 0e2c4bac8c6e)


,par,horizonte,bloque,precision_050,recall_050,f1_050,auprc,brier,brier_sin_calibrar,f1_regresion,razon_mae,razon_rmse
2,E1|S1,2,temporada_2022,0.520,0.503,0.511,0.563,0.081,0.080,0.679,0.987,1.332
2,E1|S2_pois,2,temporada_2022,0.520,0.503,0.511,0.563,0.081,0.080,0.722,1.029,1.092
2,E1|S2_twd,2,temporada_2022,0.520,0.503,0.511,0.563,0.081,0.080,0.714,1.072,1.377
2,E1|S3,2,temporada_2022,0.520,0.503,0.511,0.563,0.081,0.080,0.704,1.134,1.415
2,E1|S4_s2,2,temporada_2022,0.798,0.555,0.653,0.789,0.052,0.051,0.722,1.029,1.092
...,...,...,...,...,...,...,...,...,...,...,...,...
14,E3|S3,4,temporada_2024,0.505,0.849,0.633,0.641,0.157,0.134,0.755,1.750,2.614
14,E3|S4_s2,4,temporada_2024,0.664,0.867,0.751,0.820,0.100,0.098,0.753,1.504,1.874
14,E3|S4_s3,4,temporada_2024,0.675,0.845,0.751,0.836,0.095,0.091,0.755,1.750,2.614
14,E3|S5_iso,4,temporada_2024,0.487,0.869,0.627,0.596,0.163,0.134,0.754,1.229,1.659


In [35]:
antelacion_f = pd.concat(
    {(par, s): r["antelacion_ic"].query("alerta == 'alerta_050'") for par in pares for s, r in evaluaciones_f[par].items()},
    names=["par", "semilla"]).reset_index(level=[0, 1])
antelacion_f_resumen = (antelacion_f.groupby(["par", "horizonte", "bloque"]).avisados_2_o_mas
                        .agg(mediana="median", minimo="min", maximo="max").reset_index())
antelacion_f_resumen.query("bloque == 'seleccion_2022_2023'").pivot_table(index="par", columns="horizonte", values="mediana")

horizonte,2,3,4
par,,,
E1|S1,0.116,0.095,0.084
E1|S2_pois,0.116,0.095,0.084
E1|S2_twd,0.116,0.095,0.084
E1|S3,0.116,0.095,0.084
E1|S4_s2,0.095,0.084,0.126
E1|S4_s3,0.116,0.147,0.137
E1|S5_iso,0.116,0.116,0.074
E1|S5_spw,0.063,0.063,0.053
E3|S1,0.095,0.095,0.095


In [36]:
coherencia_f = eh.resumen_coherencia({par: [eh.coherencia_horizontes(r["datos"], serie) for r in evaluaciones_f[par].values()]
                                     for par in pares})
coherencia_f.query("bloque in @pa.BLOQUES_SELECCION").pivot_table(index="estrategia", columns="bloque", values="cambio_sin_explicar")

bloque,temporada_2022,temporada_2023
estrategia,,
E1|S1,0.016,0.023
E1|S2_pois,0.016,0.023
E1|S2_twd,0.016,0.023
E1|S3,0.016,0.023
E1|S4_s2,0.016,0.012
E1|S4_s3,0.021,0.005
E1|S5_iso,0.024,0.051
E1|S5_spw,0.019,0.036
E3|S1,0.010,0.018


In [37]:
# Errores por fase de la curva (el RMSE no decide solo): mediana de semillas POR TEMPORADA (no se mezclan 2022 y 2023).
fases_f = pd.concat({(par, s): r["errores_por_fase"] for par in pares for s, r in evaluaciones_f[par].items()},
                    names=["par", "semilla"]).reset_index(level=[0, 1])
fases_f = (fases_f[fases_f.bloque.isin(pa.BLOQUES_SELECCION)].astype({"fase": str})
           .groupby(["par", "horizonte", "bloque", "fase"])[["razon_mae", "razon_rmse", "sesgo_modelo"]].median().reset_index())
fases_f.pivot_table(index=["par", "bloque"], columns=["horizonte", "fase"], values="razon_mae").round(2)

horizonte                      2                         3                     \
fase                      bajada  pico subida valle bajada  pico subida valle   
par        bloque                                                               
E1|S1      temporada_2022   0.55  1.16   1.26  0.79   0.42  1.35   1.21  0.78   
           temporada_2023   0.63  1.20   0.95  1.00   0.50  1.11   0.92  0.93   
E1|S2_pois temporada_2022   0.69  1.21   1.10  0.95   0.66  1.62   1.10  0.90   
           temporada_2023   0.79  1.22   0.91  1.06   0.52  1.11   0.85  0.96   
E1|S2_twd  temporada_2022   0.58  1.33   1.25  0.82   0.47  1.80   1.30  0.79   
           temporada_2023   0.59  1.16   0.95  0.99   0.46  1.09   0.95  0.89   
E1|S3      temporada_2022   0.58  1.43   1.32  0.87   0.44  1.97   1.42  0.81   
           temporada_2023   0.55  1.03   1.00  1.00   0.40  1.03   1.03  0.92   
E1|S4_s2   temporada_2022   0.69  1.21   1.10  0.95   0.66  1.62   1.10  0.90   
           temporada_2023   0.79  1.22   0.91  1.06   0.52  1.11   0.85  0.96   
E1|S4_s3   temporada_2022   0.58  1.43   1.32  0.87   0.44  1.97   1.42  0.81   
           temporada_2023   0.55  1.03   1.00  1.00   0.40  1.03   1.03  0.92   
E1|S5_iso  temporada_2022   0.55  1.16   1.26  0.79   0.42  1.35   1.21  0.78   
           temporada_2023   0.63  1.20   0.95  1.00   0.50  1.11   0.92  0.93   
E1|S5_spw  temporada_2022   0.55  1.16   1.26  0.79   0.42  1.35   1.21  0.78   
           temporada_2023   0.63  1.20   0.95  1.00   0.50  1.11   0.92  0.93   
E3|S1      temporada_2022   0.57  1.13   1.31  0.80   0.43  1.35   1.25  0.80   
           temporada_2023   0.67  1.20   1.00  0.99   0.50  1.12   0.92  0.91   
E3|S2_pois temporada_2022   0.74  1.21   1.12  0.97   0.69  1.58   1.11  0.94   
           temporada_2023   0.69  1.22   0.93  1.02   0.52  1.12   0.84  0.95   
E3|S2_twd  temporada_2022   0.61  1.29   1.30  0.82   0.49  1.77   1.28  0.80   
           temporada_2023   0.60  1.15   1.05  0.99   0.46  1.08   0.95  0.89   
E3|S3      temporada_2022   0.58  1.45   1.30  0.88   0.44  2.00   1.30  0.83   
           temporada_2023   0.54  1.03   1.04  0.99   0.41  1.03   1.04  0.93   
E3|S4_s2   temporada_2022   0.74  1.21   1.12  0.97   0.69  1.58   1.11  0.94   
           temporada_2023   0.69  1.22   0.93  1.02   0.52  1.12   0.84  0.95   
E3|S4_s3   temporada_2022   0.58  1.45   1.30  0.88   0.44  2.00   1.30  0.83   
           temporada_2023   0.54  1.03   1.04  0.99   0.41  1.03   1.04  0.93   
E3|S5_iso  temporada_2022   0.57  1.13   1.31  0.80   0.43  1.35   1.25  0.80   
           temporada_2023   0.67  1.20   1.00  0.99   0.50  1.12   0.92  0.91   
E3|S5_spw  temporada_2022   0.57  1.13   1.31  0.80   0.43  1.35   1.25  0.80   
           temporada_2023   0.67  1.20   1.00  0.99   0.50  1.12   0.92  0.91   

horizonte                      4                     
fase                      bajada  pico subida valle  
par        bloque                                    
E1|S1      temporada_2022   0.39  1.72   1.13  0.68  
           temporada_2023   0.43  1.11   0.90  0.85  
E1|S2_pois temporada_2022   0.68  2.02   1.02  0.77  
           temporada_2023   0.42  1.10   0.81  0.88  
E1|S2_twd  temporada_2022   0.47  2.36   1.15  0.68  
           temporada_2023   0.40  1.08   0.88  0.85  
E1|S3      temporada_2022   0.37  2.56   1.30  0.70  
           temporada_2023   0.34  1.08   0.95  0.91  
E1|S4_s2   temporada_2022   0.68  2.02   1.02  0.77  
           temporada_2023   0.42  1.10   0.81  0.88  
E1|S4_s3   temporada_2022   0.37  2.56   1.30  0.70  
           temporada_2023   0.34  1.08   0.95  0.91  
E1|S5_iso  temporada_2022   0.39  1.72   1.13  0.68  
           temporada_2023   0.43  1.11   0.90  0.85  
E1|S5_spw  temporada_2022   0.39  1.72   1.13  0.68  
           temporada_2023   0.43  1.11   0.90  0.85  
E3|S1      temporada_2022   0.37  1.74   1.13  0.75  
           temporada_2023   0.42  1.11   0.89  0.85  
E3|S2_pois temporada_2022   0.

In [38]:
conteos_f = pd.concat([sp.comparar_conteos(medianas_f, rangos_f, e) for e in ("E1", "E3")], ignore_index=True)
conteos_f

,estrategia,conteo,horizonte,razon_mae_temporada_2022,razon_mae_temporada_2023,razon_rmse_temporada_2022,razon_rmse_temporada_2023,mejora_en_ambas,empeora_en_alguna,cumple_085_en_ambas
0,E1,log1p,2,0.987,0.986,1.332,1.140,False,False,False
1,E1,log1p,3,1.009,0.865,1.572,0.956,False,False,False
2,E1,log1p,4,1.089,0.805,1.919,0.859,False,False,False
3,E1,poisson,2,1.029,1.031,1.092,1.193,False,True,False
4,E1,poisson,3,1.156,0.868,1.372,0.963,False,True,False
5,E1,poisson,4,1.258,0.786,1.698,0.854,False,True,False
6,E1,tweedie,2,1.072,0.952,1.377,1.102,False,True,False
7,E1,tweedie,3,1.203,0.844,1.783,0.937,False,True,False
8,E1,tweedie,4,1.342,0.779,2.187,0.844,False,True,False
9,E1,poisson_margen,2,1.134,0.886,1.415,0.994,False,True,False


### Selección preregistrada dentro de cada estrategia (frente a S1)

In [39]:
seleccion = {e: sp.elegir(evaluaciones_f, [sp.clave(e, f) for f in sp.FORMULACIONES if f != "S1"],
                         sp.clave(e, "S1"), coherencia_f) for e in ("E1", "E3")}
for e, s in seleccion.items():
    print(e, s["decision"])
    display(s["adopcion"])
pd.concat([s["veredictos"] for s in seleccion.values()], ignore_index=True)

E1 {'elegida': ['E1|S4_s2', 'E1|S4_s3'], 'criterio': 'empate sin resolver por simplicidad', 'empatadas': ['E1|S4_s2', 'E1|S4_s3']}


,candidata,gana_en,pierde_en,resultado
0,E1|S2_pois,NaN,NaN,empate
1,E1|S2_twd,NaN,NaN,empate
2,E1|S3,NaN,NaN,empate
3,E1|S4_s2,"h=3, h=4",NaN,adoptable
4,E1|S4_s3,"h=2, h=3",NaN,adoptable
5,E1|S5_iso,NaN,NaN,empate
6,E1|S5_spw,NaN,"h=2, h=3, h=4",descartada


E3 {'elegida': ['E3|S4_s2', 'E3|S4_s3'], 'criterio': 'empate sin resolver por simplicidad', 'empatadas': ['E3|S4_s2', 'E3|S4_s3']}


,candidata,gana_en,pierde_en,resultado
0,E3|S2_pois,NaN,NaN,empate
1,E3|S2_twd,NaN,NaN,empate
2,E3|S3,NaN,NaN,empate
3,E3|S4_s2,"h=2, h=3, h=4",NaN,adoptable
4,E3|S4_s3,"h=2, h=3, h=4",NaN,adoptable
5,E3|S5_iso,NaN,NaN,empate
6,E3|S5_spw,NaN,"h=2, h=3, h=4",descartada


,horizonte,veredicto,mejora_f1_050,empeora_f1_050,mejora_antelacion,empeora_antelacion,vetos,vetos_contra_referencia,umbral_f1_050_temporada_2022,umbral_f1_050_temporada_2023,...,delta_brier_temporada_2022,delta_brier_temporada_2023,delta_razon_mae_temporada_2022,delta_razon_mae_temporada_2023,candidata,referencia,antelacion_diferencia,antelacion_ic95_inferior,antelacion_ic95_superior,antelacion_signo_mediana_antelacion
0,2,empate,False,False,False,False,razon_mae,None,0.021,0.014,...,0.000e+00,0.000,0.042,0.044,E1|S2_pois,E1|S1,0.000,0.000,0.000,0.0
1,3,empate,False,False,False,False,NaN,None,0.005,0.014,...,0.000e+00,0.000,0.147,0.003,E1|S2_pois,E1|S1,0.000,0.000,0.000,0.0
2,4,empate,False,False,False,False,NaN,None,0.013,0.024,...,0.000e+00,0.000,0.169,-0.019,E1|S2_pois,E1|S1,0.000,0.000,0.000,0.0
3,2,empate,False,False,False,False,None,None,0.021,0.014,...,0.000e+00,0.000,0.085,-0.034,E1|S2_twd,E1|S1,0.000,0.000,0.000,0.0
4,3,empate,False,False,False,False,None,None,0.005,0.014,...,0.000e+00,0.000,0.194,-0.021,E1|S2_twd,E1|S1,0.000,0.000,0.000,0.0
5,4,empate,False,False,False,False,None,None,0.013,0.024,...,0.000e+00,0.000,0.253,-0.026,E1|S2_twd,E1|S1,0.000,0.000,0.000,0.0
6,2,empate,False,False,False,False,None,None,0.021,0.014,...,0.000e+00,0.000,0.147,-0.100,E1|S3,E1|S1,0.000,0.000,0.000,0.0
7,3,empate,False,False,False,False,None,None,0.005,0.014,...,0.000e+00,0.000,0.276,-0.053,E1|S3,E1|S1,0.000,0.000,0.000,0.0
8,4,empate,False,False,False,False,None,None,0.013,0.024,...,0.000e+00,0.000,0.340,-0.035,E1|S3,E1|S1,0.000,0.000,0.000,0.0
9,2,compensa (empate),True,False,False,False,razon_mae,"auprc, brier",0.021,0.014,...,-2.875e-02,-0.041,0.042,0.044,E1|S4_s2,E1|S1,-0.011,-0.074,0.053,-1.0


### Mejor par de E3 frente a mejor par de E1

In [40]:
mejor = {e: s["decision"]["elegida"] for e, s in seleccion.items()}
# Si una estrategia no resuelve su formulación, el mejor par de la otra se compara con cada finalista.
finalistas_e3 = mejor["E3"] if isinstance(mejor["E3"], list) else [mejor["E3"]]
finalistas_e1 = mejor["E1"] if isinstance(mejor["E1"], list) else [mejor["E1"]]
finales = {(a, b): sp.elegir(evaluaciones_f, [a], b, coherencia_f) for a in finalistas_e3 for b in finalistas_e1}
for (a, b), r in finales.items():
    print(a, "frente a", b, "→", r["decision"])
    display(r["adopcion"])
pd.concat([r["veredictos"] for r in finales.values()], ignore_index=True)

E3|S4_s2 frente a E1|S4_s2 → {'elegida': 'E3|S4_s2', 'criterio': 'simplicidad operativa', 'empatadas': ['E1|S4_s2', 'E3|S4_s2']}


,candidata,gana_en,pierde_en,resultado
0,E3|S4_s2,None,None,empate


E3|S4_s2 frente a E1|S4_s3 → {'elegida': 'E3|S4_s2', 'criterio': 'simplicidad operativa', 'empatadas': ['E1|S4_s3', 'E3|S4_s2']}


,candidata,gana_en,pierde_en,resultado
0,E3|S4_s2,None,None,empate


E3|S4_s3 frente a E1|S4_s2 → {'elegida': 'E3|S4_s3', 'criterio': 'gana según la regla preregistrada'}


,candidata,gana_en,pierde_en,resultado
0,E3|S4_s3,h=3,None,adoptable


E3|S4_s3 frente a E1|S4_s3 → {'elegida': 'E3|S4_s3', 'criterio': 'simplicidad operativa', 'empatadas': ['E1|S4_s3', 'E3|S4_s3']}


,candidata,gana_en,pierde_en,resultado
0,E3|S4_s3,None,None,empate


,horizonte,veredicto,mejora_f1_050,empeora_f1_050,mejora_antelacion,empeora_antelacion,vetos,vetos_contra_referencia,umbral_f1_050_temporada_2022,umbral_f1_050_temporada_2023,...,delta_brier_temporada_2022,delta_brier_temporada_2023,delta_razon_mae_temporada_2022,delta_razon_mae_temporada_2023,candidata,referencia,antelacion_diferencia,antelacion_ic95_inferior,antelacion_ic95_superior,antelacion_signo_mediana_antelacion
0,2,empate,False,False,False,False,None,"auprc, brier",0.019,0.004,...,-1.279e-03,-8.047e-04,0.027,-0.025,E3|S4_s2,E1|S4_s2,-0.011,-0.032,0.000,-1.0
1,3,empate,False,False,False,False,None,auprc,0.009,0.005,...,-1.197e-03,8.291e-05,-0.003,-0.003,E3|S4_s2,E1|S4_s2,0.000,-0.032,0.032,1.0
2,4,empate,False,False,False,False,None,NaN,0.021,0.008,...,-1.438e-03,1.820e-03,0.036,0.007,E3|S4_s2,E1|S4_s2,0.000,-0.053,0.053,-1.0
3,2,empate,False,False,False,False,None,brier,0.019,0.004,...,-1.832e-03,-1.366e-03,-0.078,0.120,E3|S4_s2,E1|S4_s3,-0.032,-0.074,0.000,-1.0
4,3,empate,False,False,False,False,None,"auprc, brier",0.008,0.005,...,-2.583e-03,-8.170e-04,-0.133,0.053,E3|S4_s2,E1|S4_s3,-0.053,-0.105,0.000,-1.0
5,4,empate,False,False,False,False,None,auprc,0.043,0.008,...,-3.756e-03,1.821e-04,-0.135,0.023,E3|S4_s2,E1|S4_s3,-0.032,-0.095,0.032,-1.0
6,2,empate,False,False,False,False,None,None,0.013,0.003,...,2.937e-04,7.307e-04,0.104,-0.139,E3|S4_s3,E1|S4_s2,0.011,-0.021,0.042,1.0
7,3,gana,True,False,False,False,None,None,0.016,0.005,...,-1.219e-04,1.366e-03,0.114,-0.055,E3|S4_s3,E1|S4_s2,0.011,-0.021,0.053,1.0
8,4,empate,False,False,False,False,None,None,0.017,0.006,...,-5.320e-04,2.245e-03,0.169,-0.018,E3|S4_s3,E1|S4_s2,0.042,-0.011,0.105,1.0
9,2,empate,False,False,False,False,None,NaN,0.013,0.003,...,-2.589e-04,1.695e-04,-0.001,0.005,E3|S4_s3,E1|S4_s3,-0.011,-0.032,0.000,-1.0


In [41]:
# Fiabilidad de cada formulación (probabilidad calibrada, semilla 17)
for estrategia in ("E1", "E3"):
    d.guardar_figura(sp.figura_fiabilidad_formulaciones(
        {f: evaluaciones_f[sp.clave(estrategia, f)][17]["fiabilidad"] for f in sp.FORMULACIONES},
        f"Fiabilidad por formulación, {estrategia}"), FIGURAS_ARQUITECTURA, f"fiabilidad_formulaciones_{estrategia}")

In [42]:
# Importancia por horizonte (HU0008-4) de las finalistas de E3: SHAP del modelo de la temporada 2023, semilla 17.
finalistas_shap = mejor["E3"] if isinstance(mejor["E3"], list) else [mejor["E3"]]
shap_e3 = {}
for par in finalistas_shap:
    _, modelos_par = sp.predecir_e3(matrices, 17, par.split("|")[1], devolver_modelos=True)
    m23 = modelos_par["temporada_2023"]
    shap_e3[f"{par} regresor"] = sp.importancia_shap_por_horizonte(m23["regresor"], m23["x_reg"])
    if m23["clasificador"] is not None:
        shap_e3[f"{par} clasificador"] = sp.importancia_shap_por_horizonte(m23["clasificador"], m23["x_cls"])
pd.concat(shap_e3, names=["modelo", "horizonte"])

hist_lag_a  hist_lag_b  hist_media_4  \
modelo            horizonte                                         
E3|S4_s2 regresor h=2             1.406       0.297         1.000   
                  h=3             1.351       0.285         0.954   
                  h=4             1.300       0.272         0.910   
E3|S4_s3 regresor h=2             0.094       0.197         1.120   
                  h=3             0.109       0.170         1.011   
                  h=4             0.117       0.141         0.899   

                             hist_semanas_pos_4  semana_epi_seno  \
modelo            horizonte                                        
E3|S4_s2 regresor h=2                     0.107            0.189   
                  h=3                     0.100            0.257   
                  h=4                     0.093            0.308   
E3|S4_s3 regresor h=2                     0.255            0.392   
                  h=3                     0.234            0.465   
                  h=4                     0.214            0.516   

                             semana_epi_coseno  
modelo            horizonte                     
E3|S4_s2 regresor h=2                    0.076  
                  h=3                    0.127  
                  h=4                    0.182  
E3|S4_s3 regresor h=2                    0.099  
                  h=3                    0.203  
                  h=4                    0.324

In [43]:
destino4 = EXPERIMENTOS_ARQUITECTURA / "fase4"
destino4.mkdir(parents=True, exist_ok=True)
for par, por_semilla in predicciones_f.items():
    pd.concat(por_semilla.values(), ignore_index=True).to_csv(destino4 / f"predicciones_{par.replace('|', '_')}.csv.gz", index=False)
reporte_f = eh.reporte_mlflow("fase4_formulaciones_mediana_semillas",
                              {par: medianas_f[medianas_f.par.eq(par)] for par in pares}, pa.SEMILLAS)
registrar_o_avisar("arquitectura_horizontes_fase4_formulaciones", reporte_f, metricas_path=METRICAS_ARQUITECTURA,
                   salida_dir=destino4, gold_dir=None)
ruta = d.guardar_seccion_metricas(METRICAS_ARQUITECTURA, "fase4_formulaciones", {
    "formulaciones": sp.FORMULACIONES, "modelos_por_par": {sp.clave(*k): v for k, v in sp.MODELOS.items()},
    "compatibilidad_multi_output_tree": d.a_json(compatibilidad),
    "metricas_mediana_semillas": d.a_json(tabla_f),
    "rangos_semillas": d.a_json(rangos_f.query("bloque in @pa.BLOQUES_SELECCION")),
    "antelacion": d.a_json(antelacion_f_resumen),
    "coherencia": d.a_json(coherencia_f),
    "errores_por_fase_seleccion": d.a_json(fases_f),
    "conteos_frente_a_log1p": d.a_json(conteos_f),
    "seleccion_por_estrategia": {e: {"veredictos": d.a_json(s["veredictos"]), "adopcion": d.a_json(s["adopcion"]),
                                     "cruces": d.a_json(s["cruces"]) if len(s["cruces"]) else [],
                                     "decision": s["decision"]} for e, s in seleccion.items()},
    "final_e3_frente_a_e1": [{"candidata": a, "referencia": b, "veredictos": d.a_json(r["veredictos"]),
                              "adopcion": d.a_json(r["adopcion"]), "decision": r["decision"]}
                             for (a, b), r in finales.items()],
    "shap_por_horizonte_e3_temporada_2023": {k: d.a_json(v.reset_index(names="horizonte")) for k, v in shap_e3.items()},
})
print(ruta.relative_to(ROOT))

MLflow: run 8e561f1be646404faae98a0c5dfe277f en 'denguekay/arquitectura_horizontes_fase4_formulaciones' (sqlite:///C:/Users/Rosa/RufoEsEternoElRegreso/Denguekay/mlflow.db)
docs\modeling\metricas\arquitectura_horizontes.json


### Complementos de la fase 4

**Parámetros de S4 y no fuga.** La media de casos sale del modelo de conteo del bloque. La dispersión α de la binomial negativa se estima por máxima verosimilitud con las predicciones **dentro del entrenamiento del mismo bloque** (`dispersion_binomial_negativa(predecir_conteo(reg, train…), train.casos_Dengue)`). El umbral `k` viene de `umbral_brote_casos`, con su propio test de no fuga, y el calibrador Platt es el cronológico del protocolo. `test_s4_parametros_solo_con_entrenamiento_del_bloque` altera toda etiqueta que cierra después del corte de 2022 de cada horizonte y comprueba que las probabilidades S4 de 2022 no cambian (E1 y E3).

In [44]:
def _calcular_fase4_poisson():
    # Poisson frente a binomial negativa (diagnóstico: mismas medias, distinta distribución).
    variantes_s4 = {}
    for par in [p for p in pares if "S4" in p]:
        for semilla in pa.SEMILLAS:
            nb_pred = predicciones_f[par][semilla]
            pois_pred = nb_pred.assign(probabilidad_brote=sp.probabilidad_poisson(nb_pred.casos_predichos, nb_pred.umbral_brote_casos))
            variantes_s4[(par, "binomial_negativa", semilla)] = evaluaciones_f[par][semilla]
            variantes_s4[(par, "poisson", semilla)] = pa.evaluar(pois_pred, serie)
    return variantes_s4

variantes_s4 = cache.cargar_o_calcular("fase4_poisson", _calcular_fase4_poisson, huella=HUELLA, forzar=FORZAR_REENTRENO)
s4_dist = pd.concat({k: v["metricas"] for k, v in variantes_s4.items()}, names=["par", "distribucion", "semilla"]).reset_index(level=[0, 1, 2])
s4_dist = (s4_dist[s4_dist.bloque.isin([*pa.BLOQUES_SELECCION, pa.BLOQUE_CONTRASTE])]
           .groupby(["par", "distribucion", "horizonte", "bloque"])[
               ["brier_sin_calibrar", "brier", "prevalencia", "prob_media_calibrada", "f1_050", "auprc"]].median().reset_index())
# Probabilidad media SIN calibrar frente a la prevalencia (sobreconfianza = cola demasiado delgada).
cruda = pd.concat({k: v["datos"].groupby(["horizonte", "bloque"]).probabilidad_brote.mean() for k, v in variantes_s4.items()},
                  names=["par", "distribucion", "semilla"]).rename("prob_media_sin_calibrar").reset_index()
cruda = cruda.groupby(["par", "distribucion", "horizonte", "bloque"]).prob_media_sin_calibrar.median().reset_index()
s4_dist = s4_dist.merge(cruda, on=["par", "distribucion", "horizonte", "bloque"])
s4_dist.sort_values(["par", "horizonte", "bloque", "distribucion"])

caché: 'fase4_poisson' recalculada (forzada) (huella 0e2c4bac8c6e)


,par,distribucion,horizonte,bloque,brier_sin_calibrar,brier,prevalencia,prob_media_calibrada,f1_050,auprc,prob_media_sin_calibrar
0,E1|S4_s2,binomial_negativa,2,temporada_2022,0.051,0.052,0.128,0.120,0.653,0.789,0.128
9,E1|S4_s2,poisson,2,temporada_2022,0.055,0.055,0.128,0.112,0.619,0.782,0.146
1,E1|S4_s2,binomial_negativa,2,temporada_2023,0.064,0.064,0.307,0.265,0.847,0.946,0.264
10,E1|S4_s2,poisson,2,temporada_2023,0.060,0.066,0.307,0.270,0.827,0.945,0.305
2,E1|S4_s2,binomial_negativa,2,temporada_2024,0.072,0.074,0.254,0.289,0.811,0.890,0.260
...,...,...,...,...,...,...,...,...,...,...,...
69,E3|S4_s3,poisson,4,temporada_2022,0.065,0.067,0.128,0.090,0.507,0.721,0.136
61,E3|S4_s3,binomial_negativa,4,temporada_2023,0.099,0.089,0.307,0.232,0.785,0.916,0.200
70,E3|S4_s3,poisson,4,temporada_2023,0.082,0.091,0.307,0.245,0.741,0.912,0.275
62,E3|S4_s3,binomial_negativa,4,temporada_2024,0.091,0.095,0.254,0.313,0.751,0.836,0.222


In [45]:
fiab_cruda = []
for (par, dist, semilla), r in variantes_s4.items():
    if semilla != 17:
        continue
    datos_s4 = r["datos"][r["datos"].bloque.isin([*pa.BLOQUES_SELECCION, pa.BLOQUE_CONTRASTE])]
    for (h, bloque), g in datos_s4.groupby(["horizonte", "bloque"]):
        fiab_cruda.append(d.curva_fiabilidad(g.brote, g.probabilidad_brote).reset_index(names="bin")
                          .assign(par=par, distribucion=dist, horizonte=h, bloque=bloque))
    if par.startswith("E3") and par.endswith("S4_s2"):
        d.guardar_figura(d.figura_fiabilidad(datos_s4), FIGURAS_ARQUITECTURA, f"fiabilidad_s4_{dist}_sin_calibrar_E3")
fiab_cruda = pd.concat(fiab_cruda, ignore_index=True)
fiab_cruda.query("par == 'E3|S4_s2'").pivot_table(index=["horizonte", "bin"], columns=["distribucion", "bloque"],
                                                  values="frecuencia_observada").round(2)

distribucion         binomial_negativa                                \
bloque                  temporada_2022 temporada_2023 temporada_2024   
horizonte bin                                                          
2         [0.0, 0.1)              0.01           0.02           0.01   
          [0.1, 0.2)              0.08           0.19           0.07   
          [0.2, 0.3)              0.21           0.28           0.13   
          [0.3, 0.4)              0.36           0.46           0.25   
          [0.4, 0.5)              0.53           0.58           0.43   
          [0.5, 0.6)              0.60           0.65           0.53   
          [0.6, 0.7)              0.73           0.87           0.73   
          [0.7, 0.8)              0.82           0.95           0.82   
          [0.8, 0.9)              0.85           0.98           0.92   
          [0.9, 1.0)              0.99           1.00           0.99   
3         [0.0, 0.1)              0.01           0.03           0.01   
          [0.1, 0.2)              0.09           0.20           0.08   
          [0.2, 0.3)              0.21           0.31           0.13   
          [0.3, 0.4)              0.40           0.53           0.28   
          [0.4, 0.5)              0.55           0.69           0.46   
          [0.5, 0.6)              0.57           0.84           0.65   
          [0.6, 0.7)              0.80           0.92           0.82   
          [0.7, 0.8)              0.92           0.97           0.90   
          [0.8, 0.9)              0.95           0.99           0.98   
          [0.9, 1.0)              1.00           1.00           1.00   
4         [0.0, 0.1)              0.01           0.03           0.01   
          [0.1, 0.2)              0.12           0.19           0.08   
          [0.2, 0.3)              0.24           0.39           0.16   
          [0.3, 0.4)              0.45           0.60           0.35   
          [0.4, 0.5)              0.55           0.81           0.54   
          [0.5, 0.6)              0.71           0.93           0.74   
          [0.6, 0.7)              0.85           0.96           0.87   
          [0.7, 0.8)              0.95           0.99           0.93   
          [0.8, 0.9)              1.00           1.00           1.00   
          [0.9, 1.0)              1.00            NaN            NaN   

distribucion                poisson                                
bloque               temporada_2022 temporada_2023 temporada_2024  
horizonte bin                                                      
2         [0.0, 0.1)           0.02           0.03           0.01  
          [0.1, 0.2)           0.10           0.15           0.11  
          [0.2, 0.3)           0.23           0.26           0.12  
          [0.3, 0.4)           0.30           0.43           0.13  
          [0.4, 0.5)           0.44           0.50           0.34  
          [0.5, 0.6)           0.43           0.53           0.33  
          [0.6, 0.7)           0.64           0.61           0.43  
          [0.7, 0.8)           0.68           0.58           0.46  
          [0.8, 0.9)           0.63           0.79           0.59  
          [0.9, 1.0)           0.83           0.98           0.87  
3         [0.0, 0.1)           0.02           0.04           0.01  
          [0.1, 0.2)           0.11           0.21           0.11  
          [0.2, 0.3)           0.24           0.30           0.13  
          [0.3, 0.4)           0.28           0.36           0.17  
          [0.4, 0.5)           0.44           0.43           0.17  
          [0.5, 0.6)           0.37           0.51           0.37  
          [0.6, 0.7)           0.36           0.67           0.38  
          [0.7, 0.8)           0.62           0.77           0.44  
          [0.8, 0.9)           0.66           0.80           0.57  
          [0.9, 1.0)           0.81           0.96           0.82  
4         [0.0, 0.1)           0.02           0.05         

In [46]:
# Antelación de cada par frente a la persistencia (inicios limpios 2022–2023, diferencia pareada, semilla 17).
ant_pers = []
for par in pares:
    t = evaluaciones_f[par][17]["antelacion"]
    dif = pa.diferencia_antelacion(t, t, "alerta_050", "alerta_persistencia").assign(par=par)
    med = antelacion_f.query("par == @par and bloque == 'seleccion_2022_2023'").groupby("horizonte").avisados_2_o_mas.median()
    pers = evaluaciones_f[par][17]["antelacion_ic"].query("alerta == 'alerta_persistencia' and bloque == 'seleccion_2022_2023'")
    ant_pers.append(dif.merge(med.rename("modelo_mediana_semillas"), left_on="horizonte", right_index=True)
                    .merge(pers[["horizonte", "avisados_2_o_mas"]].rename(columns={"avisados_2_o_mas": "persistencia"}), on="horizonte"))
ant_pers = pd.concat(ant_pers, ignore_index=True)
ant_pers[["par", "horizonte", "episodios", "modelo_mediana_semillas", "persistencia", "diferencia", "ic95_inferior", "ic95_superior"]]

,par,horizonte,episodios,modelo_mediana_semillas,persistencia,diferencia,ic95_inferior,ic95_superior
0,E1|S1,2,95,0.116,0.116,-0.011,-0.084,0.053
1,E1|S1,3,95,0.095,0.137,-0.042,-0.106,0.021
2,E1|S1,4,95,0.084,0.211,-0.126,-0.211,-0.042
3,E1|S2_pois,2,95,0.116,0.116,-0.011,-0.084,0.053
4,E1|S2_pois,3,95,0.095,0.137,-0.042,-0.106,0.021
5,E1|S2_pois,4,95,0.084,0.211,-0.126,-0.211,-0.042
6,E1|S2_twd,2,95,0.116,0.116,-0.011,-0.084,0.053
7,E1|S2_twd,3,95,0.095,0.137,-0.042,-0.106,0.021
8,E1|S2_twd,4,95,0.084,0.211,-0.126,-0.211,-0.042
9,E1|S3,2,95,0.116,0.116,-0.011,-0.084,0.053


In [47]:
# ¿Anticipa brotes nuevos o acierta más en brotes en curso? Recall por tipo de semana brote (mediana de semillas).
recall_tipo = pd.concat({(par, s): sp.recall_por_tipo_semana(r["datos"], serie) for par in pares for s, r in evaluaciones_f[par].items()},
                        names=["par", "semilla"]).reset_index(level=[0, 1])
recall_tipo = (recall_tipo[recall_tipo.bloque.isin(pa.BLOQUES_SELECCION)]
               .groupby(["par", "horizonte", "bloque", "tipo"]).agg(semanas=("semanas", "first"), recall=("recall", "median")).reset_index())
recall_tipo.pivot_table(index="par", columns=["horizonte", "bloque", "tipo"], values="recall").round(3)

horizonte               2                                               3  \
bloque     temporada_2022          temporada_2023          temporada_2022   
tipo          brote_nuevo en_curso    brote_nuevo en_curso    brote_nuevo   
par                                                                         
E1|S1               0.061    0.560          0.081    0.608          0.061   
E1|S2_pois          0.061    0.560          0.081    0.608          0.061   
E1|S2_twd           0.061    0.560          0.081    0.608          0.061   
E1|S3               0.061    0.560          0.081    0.608          0.061   
E1|S4_s2            0.061    0.618          0.105    0.849          0.020   
E1|S4_s3            0.102    0.683          0.093    0.850          0.061   
E1|S5_iso           0.061    0.644          0.070    0.580          0.143   
E1|S5_spw           0.061    0.469          0.058    0.437          0.041   
E3|S1               0.061    0.539          0.070    0.620          0.061   
E3|S2_pois          0.061    0.539          0.070    0.620          0.061   
E3|S2_twd           0.061    0.539          0.070    0.620          0.061   
E3|S3               0.061    0.539          0.070    0.620          0.061   
E3|S4_s2            0.061    0.670          0.093    0.844          0.041   
E3|S4_s3            0.082    0.678          0.093    0.854          0.061   
E3|S5_iso           0.082    0.652          0.058    0.553          0.122   
E3|S5_spw           0.061    0.445          0.058    0.392          0.041   

horizonte                                                4           \
bloque              temporada_2023          temporada_2022            
tipo       en_curso    brote_nuevo en_curso    brote_nuevo en_curso   
par                                                                   
E1|S1         0.429          0.081    0.560          0.061    0.359   
E1|S2_pois    0.429          0.081    0.560          0.061    0.359   
E1|S2_twd     0.429          0.081    0.560          0.061    0.359   
E1|S3         0.429          0.081    0.560          0.061    0.359   
E1|S4_s2      0.466          0.070    0.775          0.000    0.327   
E1|S4_s3      0.450          0.116    0.799          0.000    0.264   
E1|S5_iso     0.620          0.081    0.435          0.082    0.482   
E1|S5_spw     0.296          0.070    0.233          0.041    0.207   
E3|S1         0.435          0.081    0.551          0.041    0.356   
E3|S2_pois    0.435          0.081    0.551          0.041    0.356   
E3|S2_twd     0.435          0.081    0.551          0.041    0.356   
E3|S3         0.435          0.081    0.551          0.041    0.356   
E3|S4_s2      0.500          0.093    0.772          0.000    0.369   
E3|S4_s3      0.476          0.081    0.796          0.000    0.330   
E3|S5_iso     0.563          0.070    0.459          0.143    0.542   
E3|S5_spw     0.306          0.058    0.254          0.041    0.230   

horizonte                           
bloque     temporada_2023           
tipo          brote_nuevo en_curso  
par                                 
E1|S1               0.081    0.434  
E1|S2_pois          0.081    0.434  
E1|S2_twd           0.081    0.434  
E1|S3               0.081    0.434  
E1|S4_s2            0.105    0.735  
E1|S4_s3            0.128    0.740  
E1|S5_iso           0.035    0.204  
E1|S5_spw           0.035    0.148  
E3|S1               0.070    0.446  
E3|S2_pois          0.070    0.446  
E3|S2_twd           0.070    0.446  
E3|S3               0.070    0.446  
E3|S4_s2            0.081    0.717  
E3|S4_s3            0.151    0.731  
E3|S5_iso           0.047    0.264  
E3|S5_spw           0.047    0.169

In [48]:
# Conteo de S4 (= su modelo de casos) frente a la persistencia por fase de la curva; S1 como referencia.
fases_s4 = fases_f[fases_f.par.str.contains("S1|S4")]
fases_s4.pivot_table(index=["par", "horizonte", "bloque"], columns="fase", values=["razon_mae", "razon_rmse"]).round(2)

razon_mae                    razon_rmse  \
fase                                 bajada  pico subida valle     bajada   
par      horizonte bloque                                                   
E1|S1    2         temporada_2022      0.55  1.16   1.26  0.79       0.51   
                   temporada_2023      0.63  1.20   0.95  1.00       0.58   
         3         temporada_2022      0.42  1.35   1.21  0.78       0.41   
                   temporada_2023      0.50  1.11   0.92  0.93       0.48   
         4         temporada_2022      0.39  1.72   1.13  0.68       0.41   
                   temporada_2023      0.43  1.11   0.90  0.85       0.41   
E1|S4_s2 2         temporada_2022      0.69  1.21   1.10  0.95       0.63   
                   temporada_2023      0.79  1.22   0.91  1.06       0.73   
         3         temporada_2022      0.66  1.62   1.10  0.90       0.68   
                   temporada_2023      0.52  1.11   0.85  0.96       0.48   
         4         temporada_2022      0.68  2.02   1.02  0.77       0.72   
                   temporada_2023      0.42  1.10   0.81  0.88       0.38   
E1|S4_s3 2         temporada_2022      0.58  1.43   1.32  0.87       0.49   
                   temporada_2023      0.55  1.03   1.00  1.00       0.46   
         3         temporada_2022      0.44  1.97   1.42  0.81       0.38   
                   temporada_2023      0.40  1.03   1.03  0.92       0.35   
         4         temporada_2022      0.37  2.56   1.30  0.70       0.34   
                   temporada_2023      0.34  1.08   0.95  0.91       0.30   
E3|S1    2         temporada_2022      0.57  1.13   1.31  0.80       0.53   
                   temporada_2023      0.67  1.20   1.00  0.99       0.64   
         3         temporada_2022      0.43  1.35   1.25  0.80       0.41   
                   temporada_2023      0.50  1.12   0.92  0.91       0.48   
         4         temporada_2022      0.37  1.74   1.13  0.75       0.36   
                   temporada_2023      0.42  1.11   0.89  0.85       0.41   
E3|S4_s2 2         temporada_2022      0.74  1.21   1.12  0.97       0.68   
                   temporada_2023      0.69  1.22   0.93  1.02       0.62   
         3         temporada_2022      0.69  1.58   1.11  0.94       0.71   
                   temporada_2023      0.52  1.12   0.84  0.95       0.47   
         4         temporada_2022      0.71  2.04   1.02  0.88       0.75   
                   temporada_2023      0.45  1.09   0.81  0.91       0.41   
E3|S4_s3 2         temporada_2022      0.58  1.45   1.30  0.88       0.48   
                   temporada_2023      0.54  1.03   1.04  0.99       0.45   
         3         temporada_2022      0.44  2.00   1.30  0.83       0.38   
                   temporada_2023      0.41  1.03   1.04  0.93       0.35   
         4         temporada_2022      0.39  2.57   1.17  0.78       0.38   
                   temporada_2023      0.34  1.06   0.99  0.90       0.30   

                                                      
fase                               pico subida valle  
par      horizonte bloque                             
E1|S1    2         temporada_2022  1.38   1.80  0.83  
                   temporada_2023  1.27   0.90  1.03  
         3         temporada_2022  1.90   1.85  0.79  
                   temporada_2023  1.11   0.90  0.94  
         4         temporada_2022  2.80   1.33  0.68  
                   temporada_2023  1.05   0.88  0.80  
E1|S4_s2 2         temporada_2022  1.22   1.24  0.83  
                   temporada_2023  1.32   0.89  1.06  
         3         temporada_2022  1.81   1.30  0.79  
                   temporada_2023  1.13   0.84  0.92  
         4         temporada_2022  2.45   1.10  0.65  
                   temporada_2023  1.06   0.78  0.76  
E1|S4_s3 2         temporada_2022  1.57   1.76  0.85  
                   temporada_2023  1.11   0.96  1.01  
         3         temporada_2022  2.41   2.09  0.76  
                   temporada_2023  1.06 

In [49]:
ruta = d.guardar_seccion_metricas(METRICAS_ARQUITECTURA, "fase4_complementos", {
    "parametros_s4": "media del modelo de conteo del bloque; alfa NB por máxima verosimilitud con el entrenamiento del bloque; "
                     "k desde umbral_brote_casos (test de no fuga); Platt cronológico",
    "tests_no_fuga": ["tests/test_umbral_brote_sin_fuga.py",
                      "tests/test_salida_probabilistica.py::test_s4_parametros_solo_con_entrenamiento_del_bloque",
                      "tests/test_salida_probabilistica.py::test_s4_y_s3_sin_fuga_por_perturbacion_del_futuro"],
    "poisson_frente_a_binomial_negativa": d.a_json(s4_dist),
    "fiabilidad_sin_calibrar_s4_semilla_17": d.a_json(fiab_cruda),
    "antelacion_frente_a_persistencia": d.a_json(ant_pers),
    "recall_brote_nuevo_frente_a_en_curso": d.a_json(recall_tipo),
    "errores_por_fase_s1_s4": d.a_json(fases_s4),
})
print(ruta.relative_to(ROOT))

docs\modeling\metricas\arquitectura_horizontes.json


## Fase 5: síntesis y decisión

Decisión de Rosa al cerrar la fase 4 (regla simétrica corregida): **E3 + S4_s2**. La media de casos sale de un XGBoost `count:poisson` multi-salida (h=2, 3 y 4 desde el mismo origen). La probabilidad de brote es P(casos ≥ k) con una binomial negativa, con α estimada en el entrenamiento de cada bloque, y luego Platt cronológico.

S4_s2 y S4_s3 empatan con la regla; se elige S4_s2 por ser la más simple (no depende del margen de persistencia). Quedan documentadas como alternativas S4_s3 y E2. El detalle está en `docs/modeling/arquitectura_horizontes.md`, sección «Fase 5».

In [50]:
ELEGIDA = sp.clave("E3", "S4_s2")
metricas_elegida = medianas_f[medianas_f.par.eq(ELEGIDA)].drop(columns="par")
cumplimiento = sp.cumplimiento_hu0007_4(metricas_elegida[metricas_elegida.bloque.isin([*pa.BLOQUES_SELECCION, pa.BLOQUE_CONTRASTE])])
cumplimiento

,horizonte,bloque,precision_050,recall_050,f1_050,razon_mae,razon_rmse,cumple_precision_050,cumple_recall_050,cumple_f1_050,cumple_razon_mae,cumple_razon_rmse,cumple_clasificacion,cumple_regresion
2,2,temporada_2022,0.794,0.601,0.682,1.056,1.173,True,False,False,False,False,False,False
3,2,temporada_2023,0.919,0.782,0.845,1.005,1.174,True,False,True,False,False,False,False
4,2,temporada_2024,0.750,0.894,0.815,1.349,1.780,True,True,True,False,False,True,False
7,3,temporada_2022,0.869,0.445,0.590,1.153,1.377,True,False,False,False,False,False,False
8,3,temporada_2023,0.936,0.716,0.812,0.865,0.963,True,False,True,False,False,False,False
9,3,temporada_2024,0.716,0.880,0.789,1.465,1.883,True,True,True,False,False,True,False
12,4,temporada_2022,0.890,0.327,0.478,1.293,1.666,True,False,False,False,False,False,False
13,4,temporada_2023,0.941,0.664,0.779,0.793,0.846,True,False,True,True,True,False,True
14,4,temporada_2024,0.664,0.867,0.751,1.504,1.874,True,True,True,False,False,True,False


In [51]:
# Conteos en la sensibilidad 2025 (3 positivos: la alerta no es evaluable).
metricas_elegida.query("bloque == @pa.BLOQUE_SENSIBILIDAD_CONTEOS")[["horizonte", "mae", "mae_persistencia", "razon_mae", "rmse", "rmse_persistencia", "razon_rmse"]]

,horizonte,mae,mae_persistencia,razon_mae,rmse,rmse_persistencia,razon_rmse
0,2,0.575,0.285,2.016,1.491,0.981,1.519
5,3,0.720,0.294,2.451,2.033,1.039,1.956
10,4,0.898,0.324,2.769,2.742,1.168,2.347


In [52]:
# SHAP por horizonte (HU0008-4) del modelo de casos de E3·S4_s2, que es su único modelo; semilla 17.
_, modelos_elegida = sp.predecir_e3(matrices, 17, "S4_s2", devolver_modelos=True)
shap_elegida = {bloque: sp.importancia_shap_por_horizonte(modelos_elegida[bloque]["regresor"], modelos_elegida[bloque]["x_reg"])
                for bloque in [*pa.BLOQUES_SELECCION, pa.BLOQUE_CONTRASTE]}
alphas_elegida = {bloque: modelos_elegida[bloque]["alphas"] for bloque in shap_elegida}
display(pd.Series(alphas_elegida, name="alpha binomial negativa por horizonte (h=2, 3, 4)"))
pd.concat(shap_elegida, names=["bloque", "horizonte"])

temporada_2022    [0.8484496650844178, 1.4570941838534195, 2.390...
temporada_2023    [0.7673733561074706, 1.387354256014161, 2.2219...
temporada_2024    [0.91554776668059, 1.6319828711250157, 2.46507...
Name: alpha binomial negativa por horizonte (h=2, 3, 4), dtype: object

hist_lag_a  hist_lag_b  hist_media_4  \
bloque         horizonte                                         
temporada_2022 h=2             1.498       0.331         0.900   
               h=3             1.455       0.317         0.870   
               h=4             1.392       0.306         0.836   
temporada_2023 h=2             1.406       0.297         1.000   
               h=3             1.351       0.285         0.954   
               h=4             1.300       0.272         0.910   
temporada_2024 h=2             1.649       0.258         0.942   
               h=3             1.601       0.240         0.909   
               h=4             1.546       0.221         0.886   

                          hist_semanas_pos_4  semana_epi_seno  \
bloque         horizonte                                        
temporada_2022 h=2                     0.159            0.209   
               h=3                     0.155            0.285   
               h=4                     0.150            0.345   
temporada_2023 h=2                     0.107            0.189   
               h=3                     0.100            0.257   
               h=4                     0.093            0.308   
temporada_2024 h=2                     0.091            0.178   
               h=3                     0.087            0.253   
               h=4                     0.086            0.313   

                          semana_epi_coseno  
bloque         horizonte                     
temporada_2022 h=2                    0.078  
               h=3                    0.130  
               h=4                    0.184  
temporada_2023 h=2                    0.076  
               h=3                    0.127  
               h=4                    0.182  
temporada_2024 h=2                    0.049  
               h=3                    0.085  
               h=4                    0.117

In [53]:
ruta = d.guardar_seccion_metricas(METRICAS_ARQUITECTURA, "fase5_decision", {
    "elegida": {"estrategia": "E3", "formulacion": "S4_s2", "modelos_a_desplegar": sp.MODELOS[("E3", "S4_s2")],
                "descripcion": "XGBoost count:poisson multi_output_tree (h=2,3,4) → binomial negativa (alfa por bloque) → Platt cronológico",
                "criterio": "regla simétrica: E3·S4 gana a E1·S4 (simplicidad o regla); S4_s2 y S4_s3 empatan, Rosa elige S4_s2 por simplicidad"},
    "alternativas_documentadas": {
        "E3·S4_s3": "empata con S4_s2; mejor antelación en h=4 (no significativa); revisar si la anticipación pasa a ser el problema central",
        "E2": "empató con E3 en la fase 3; no se evaluó con S4: en h=2 empeoró AUPRC y MAE en ambas temporadas"},
    "metricas_mediana_semillas": d.a_json(metricas_elegida),
    "cumplimiento_hu0007_4": d.a_json(cumplimiento),
    "shap_por_horizonte_semilla_17": {b: d.a_json(t.reset_index(names="horizonte")) for b, t in shap_elegida.items()},
    "alpha_binomial_negativa_semilla_17": {b: list(map(float, a)) for b, a in alphas_elegida.items()},
})
print(ruta.relative_to(ROOT))

docs\modeling\metricas\arquitectura_horizontes.json
